<a href="https://colab.research.google.com/github/racoope70/exploratory-daytrading/blob/main/ppo_xgboost_hybrid_trading_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install "shimmy>=2.0.0"

In [13]:
!pip -q install yfinance pywavelets transformers --upgrade

In [14]:
!apt-get remove --purge -y cuda* libcuda* nvidia* || echo "No conflicting CUDA packages"
!apt-get autoremove -y
!apt-get clean

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'cuda-drivers-fabricmanager-450' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-460' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-470' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-510' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-515' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-525' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-535' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-550' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-565' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-570' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-575' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-580' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager-590' for glob 'cuda*'
Note, selecting 'cuda-drivers-fabricmanager' 

In [15]:
!apt-get update -qq && apt-get install -y \
    libcusolver11 libcusparse11 libcurand10 libcufft10 libnppig10 libnppc10 libnppial10 \
    cuda-toolkit-12-4

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libnppig10
E: Unable to locate package libnppc10
E: Unable to locate package libnppial10
E: Unable to locate package cuda-toolkit-12-4


In [16]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3


Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3
  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orbax-checkpoint 0.11.33 requires jax>=0.6.0, which is not installed.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.18.0 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.18.0 which is incompatible.
tensorflow-metadata 1.17.3 requires protobuf>=4.25.2

In [1]:
!pip install --extra-index-url=https://pypi.nvidia.com \
    cuml-cu12==25.2.0 cudf-cu12==25.2.0 cupy-cuda12x \
    dask-cuda==25.2.0 dask-cudf-cu12==25.2.0


Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [2]:
!pip install numba==0.60.0


In [3]:
!pip install "stable-baselines3[extra]>=2.0.0" "gymnasium>=0.29" "shimmy>=2.0.0" \
  gym-anytrading yfinance pandas numpy scikit-learn xgboost joblib


In [ ]:
#!pip install stable-baselines3[extra] gymnasium gym-anytrading yfinance xgboost joblib
#!pip install matplotlib scikit-learn pandas

In [4]:
!pip install tensorflow==2.18.0

In [5]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124


In [6]:
!pip uninstall -y jax jaxlib

# Then restart the runtime/kernel, and import TF normally:
import tensorflow as tf

In [7]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.4'
os.environ['PATH'] += ':/usr/local/cuda-12.4/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-12.4/lib64'


In [ ]:
# Step 7: authenticate with hugging face hub (optional)
# This allows for better access and avoids rate limits when downloading public models/datasets

# Authenticate with Hugging Face Hub
# Notebook_login()

In [8]:
from __future__ import annotations

# ============================================================
# CONFIG FIRST (Phase 0: reproducible + provenance)
# ============================================================
USE_SENTIMENT = False
USE_REGIME    = True
TEST_MODE     = True

INTERVAL     = "1h"
PERIOD_DAYS  = 720
FWD_HORIZON  = 10
UP_THR       = 0.02
DN_THR       = -0.02
VAL_FRACTION = 0.20

DATA_VENDOR  = "yfinance"
TIMEZONE_STR = "America/New_York"   # final dataset timezone (training expects tz-naive afterwards)

# Output filenames (local)
LOCAL_OUT   = "multi_stock_feature_engineered_dataset.csv"
LOCAL_TRAIN = "train.csv"
LOCAL_VAL   = "val.csv"
PARQ_FULL   = "features_full.parquet"
PARQ_TRAIN  = "train.parquet"
PARQ_VAL    = "val.parquet"

# ============================================================
# IMPORTS
# ============================================================
import os, gc, time, json, logging, hashlib, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import yfinance as yf

# Wavelet
import pywt

# ============================================================
# PATHS (Colab-safe + local-safe)
# ============================================================
try:
    from google.colab import drive  # type: ignore
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive"
except Exception:
    DRIVE_BASE = os.getcwd()

DRIVE_DIR = os.path.join(DRIVE_BASE, "trading_data")
os.makedirs(DRIVE_DIR, exist_ok=True)

# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True)
logging.info(f"yfinance version: {getattr(yf, '__version__', 'unknown')}")
logging.info(f"pandas version: {pd.__version__}")

# ============================================================
# MANIFEST (Phase 0: config.json-ish + ordered features + provenance)
#    IMPROVEMENT: define MANIFEST ONCE (no overwrites)
# ============================================================
MANIFEST_PATH = os.path.join(DRIVE_DIR, "artifact_manifest.json")

def _sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def _safe_json_dump(obj: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)
    os.replace(tmp, path)

MANIFEST = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "config": {
        "INTERVAL": INTERVAL,
        "PERIOD_DAYS": PERIOD_DAYS,
        "FWD_HORIZON": FWD_HORIZON,
        "UP_THR": UP_THR,
        "DN_THR": DN_THR,
        "VAL_FRACTION": VAL_FRACTION,
        "USE_REGIME": USE_REGIME,
        "USE_SENTIMENT": USE_SENTIMENT,
        "TEST_MODE": TEST_MODE,
        "DATA_VENDOR": DATA_VENDOR,
        "TIMEZONE": TIMEZONE_STR,
    },
    "versions": {
        "python": sys.version,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "yfinance": getattr(yf, "__version__", "unknown"),
    },
    "data_provenance": {},   # per ticker
    "feature_manifest": [],  # final list of feature columns (ordered later)
    "artifacts": {},         # written files + hashes
}

def manifest_record(key: str, path: str):
    MANIFEST.setdefault("artifacts", {})
    MANIFEST["artifacts"][key] = {
        "path": os.path.abspath(path),
        "sha256": _sha256_file(path) if os.path.exists(path) else None,
        "bytes": int(os.path.getsize(path)) if os.path.exists(path) else None,
        "written_utc": datetime.now(timezone.utc).isoformat(),
    }
    _safe_json_dump(MANIFEST, MANIFEST_PATH)

# write initial manifest once
_safe_json_dump(MANIFEST, MANIFEST_PATH)

# ============================================================
# UNIVERSE
# ============================================================
ticker_list = [
    "AAPL","TSLA","MSFT","GOOGL","AMZN","NVDA","META","BRK-B","JPM","JNJ",
    "XOM","V","PG","UNH","MA","HD","LLY","MRK","PEP","KO",
    "BAC","ABBV","AVGO","PFE","COST","CSCO","TMO","ABT","ACN","WMT",
    "MCD","ADBE","DHR","CRM","NKE","INTC","QCOM","NEE","AMD","TXN",
    "AMGN","UPS","LIN","PM","UNP","BMY","LOW","RTX","CVX","IBM",
    "GE","SBUX","ORCL",
]
SYMBOLS = ["AAPL", "NVDA", "MSFT"] if TEST_MODE else ticker_list

# ============================================================
# OPTIONAL SENTIMENT MODEL
# ============================================================
if USE_SENTIMENT:
    try:
        import torch
        from transformers import pipeline

        device_id = 0 if torch.cuda.is_available() else -1
        sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device_id)
        logging.info("FinBERT sentiment enabled.")
    except Exception as e:
        logging.warning(f"Could not init FinBERT; disabling sentiment. Err: {e}")
        USE_SENTIMENT = False
        sentiment_pipeline = None
else:
    sentiment_pipeline = None

# ============================================================
# HELPERS
# ============================================================
def _force_datetime_column(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure a tz-naive 'Datetime' column exists and data is sorted/deduped by time."""
    if isinstance(df.index, pd.DatetimeIndex):
        try:
            if df.index.tz is not None:
                df.index = df.index.tz_convert(None)
        except Exception:
            try:
                df.index = df.index.tz_localize(None)
            except Exception:
                pass
        df.index.name = "Datetime"
        df = df.reset_index()
    else:
        if not isinstance(df.index, pd.RangeIndex):
            df = df.reset_index()
        first = df.columns[0]
        if np.issubdtype(df[first].dtype, np.datetime64):
            df = df.rename(columns={first: "Datetime"})
        elif "Date" in df.columns:
            df["Datetime"] = pd.to_datetime(df["Date"], errors="coerce")
        elif "Datetime" not in df.columns:
            df["Datetime"] = pd.to_datetime(df[first], errors="coerce")

    if "Datetime" not in df.columns:
        raise KeyError("Failed to construct 'Datetime' column from yfinance output.")

    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
    df = (
        df.dropna(subset=["Datetime"])
          .drop_duplicates(subset=["Datetime"])
          .sort_values("Datetime")
          .reset_index(drop=True)
    )
    return df

def _normalize_ohlcv(df_in: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Normalize yfinance OHLCV columns to: Open, High, Low, Close, Adj Close, Volume."""
    import re
    df = df_in.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join([str(p) for p in col if p is not None and str(p) != ""]).strip()
            for col in df.columns
        ]

    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]

    tkr = ticker.upper().replace("-", "[- ]?")
    cleaned = {}
    for c in df.columns:
        c_up = c.upper()
        c2 = re.sub(rf"^(?:{tkr})[\s/_-]+", "", c_up)
        c2 = re.sub(rf"[\s/_-]+(?:{tkr})$", "", c2)
        cleaned[c] = c2.title()

    if any(cleaned[c] != c for c in df.columns):
        df = df.rename(columns=cleaned)

    cols_ci = {c.lower(): c for c in df.columns}
    wants = {
        "Open":      ["open"],
        "High":      ["high"],
        "Low":       ["low"],
        "Close":     ["close", "last"],
        "Adj Close": ["adj close", "adj_close", "adjclose", "adjusted close"],
        "Volume":    ["volume", "vol"],
    }

    rename_map = {}
    for desired, alts in wants.items():
        if desired.lower() in cols_ci:
            rename_map[cols_ci[desired.lower()]] = desired
            continue
        for a in alts:
            if a in cols_ci:
                rename_map[cols_ci[a]] = desired
                break

    if rename_map:
        df = df.rename(columns=rename_map)

    return df

def download_stock_data(ticker: str, interval: str = "1h", period_days: int = 720, max_retries: int = 5, sleep_base: int = 3):
    """Download intraday data via yfinance with normalization and a history() fallback."""
    period_str = f"{int(period_days)}d"

    def _postprocess(df: pd.DataFrame) -> pd.DataFrame:
        df = _normalize_ohlcv(df, ticker)
        df = _force_datetime_column(df)
        needed = {"Open", "High", "Low", "Close", "Volume"}
        missing = needed - set(df.columns)
        if missing:
            logging.debug(f"[{ticker}] columns received: {list(df.columns)}")
            raise ValueError(f"Missing OHLCV columns after normalize: {missing}")
        if "Adj Close" not in df.columns:
            df["Adj Close"] = df["Close"]
        return df

    for attempt in range(1, max_retries + 1):
        try:
            logging.info(f"[{ticker}] Attempt {attempt}: download(period={period_str}, interval={interval})")
            df = yf.download(
                tickers=ticker,
                period=period_str,
                interval=interval,
                progress=False,
                auto_adjust=False,
                group_by="column",
                threads=False,
                prepost=False,
                repair=True,
            )
            if df is None or df.empty:
                raise ValueError("Empty data from download()")
            df = _postprocess(df)
            df["Symbol"] = ticker
            logging.info(f"[{ticker}] rows: {len(df)} from {df['Datetime'].min()} to {df['Datetime'].max()}")
            return df
        except Exception as e1:
            logging.warning(f"[{ticker}] download error: {e1} | trying history() fallback")
            try:
                hist = yf.Ticker(ticker).history(
                    period=period_str,
                    interval=interval,
                    auto_adjust=False,
                    actions=False,
                )
                if hist is None or hist.empty:
                    raise ValueError("Empty data from history()")
                df = _postprocess(hist)
                df["Symbol"] = ticker
                logging.info(f"[{ticker}] (fallback) rows: {len(df)} from {df['Datetime'].min()} to {df['Datetime'].max()}")
                return df
            except Exception as e2:
                wait = sleep_base * attempt
                logging.warning(f"[{ticker}] history() error: {e2} | retrying in {wait}s")
                time.sleep(wait)

    logging.error(f"[{ticker}] Failed to download after {max_retries} attempts.")
    return None

def denoise_wavelet(series: pd.Series, wavelet: str = "db1", level: int = 2) -> pd.Series:
    s = pd.Series(series).astype(float).ffill().bfill().to_numpy()
    try:
        coeffs = pywt.wavedec(s, wavelet, mode="symmetric", level=level)
        for i in range(1, len(coeffs)):
            coeffs[i] = np.zeros_like(coeffs[i])
        rec = pywt.waverec(coeffs, wavelet, mode="symmetric")
        return pd.Series(rec[:len(s)], index=series.index)
    except Exception as e:
        logging.warning(f"Wavelet denoising failed ({e}); using raw Close.")
        return pd.Series(s, index=series.index)

def score_sentiment(texts: list[str]) -> list[float]:
    if not USE_SENTIMENT or sentiment_pipeline is None:
        return [0.0] * len(texts)
    try:
        outputs = sentiment_pipeline(texts, truncation=True, max_length=256, batch_size=32)
        scores = []
        for r in outputs:
            label = str(r["label"]).lower()
            if label == "positive":
                scores.append(+float(r["score"]))
            elif label == "negative":
                scores.append(-float(r["score"]))
            else:
                scores.append(0.0)
        return scores
    except Exception as e:
        logging.error(f"Sentiment scoring error: {e}")
        return [0.0] * len(texts)

def add_regime(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Vol20"] = df["Close"].pct_change().rolling(20).std()
    df["Ret20"] = df["Close"].pct_change(20)
    vol_hi   = (df["Vol20"] > df["Vol20"].median()).astype(int)
    trend_hi = (df["Ret20"].abs() > df["Ret20"].abs().median()).astype(int)
    df["Regime4"] = vol_hi * 2 + trend_hi
    return df

def compute_enhanced_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Symbol" not in df.columns:
        df["Symbol"] = "UNKNOWN"

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.loc[:, ~df.columns.duplicated()]

    req = {"Open", "High", "Low", "Close", "Volume", "Datetime", "Symbol"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # ---- Indicators ----
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["STD_20"] = df["Close"].rolling(20).std()
    df["Upper_Band"] = df["SMA_20"] + 2 * df["STD_20"]
    df["Lower_Band"] = df["SMA_20"] - 2 * df["STD_20"]

    df["Lowest_Low"]   = df["Low"].rolling(14).min()
    df["Highest_High"] = df["High"].rolling(14).max()
    denom = (df["Highest_High"] - df["Lowest_Low"]).replace(0, np.nan)
    df["Stoch"] = ((df["Close"] - df["Lowest_Low"]) / denom) * 100

    df["ROC"] = df["Close"].pct_change(10)
    df["OBV"] = (np.sign(df["Close"].diff()).fillna(0) * df["Volume"].fillna(0)).cumsum()

    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    sma_tp = tp.rolling(20).mean()
    md = (tp - sma_tp).abs().rolling(20).mean()
    df["CCI"] = (tp - sma_tp) / (0.015 * md)

    df["EMA_10"] = df["Close"].ewm(span=10, adjust=False).mean()
    df["EMA_50"] = df["Close"].ewm(span=50, adjust=False).mean()
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD_Line"]   = ema12 - ema26
    df["MACD_Signal"] = df["MACD_Line"].ewm(span=9, adjust=False).mean()

    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss.replace(0, np.nan))
    df["RSI"] = 100 - (100 / (1 + rs))

    tr = pd.concat(
        [
            (df["High"] - df["Low"]),
            (df["High"] - df["Close"].shift()).abs(),
            (df["Low"] - df["Close"].shift()).abs(),
        ],
        axis=1,
    ).max(axis=1)
    df["ATR"] = tr.rolling(14).mean()

    df["Volatility"] = df["Close"].pct_change().rolling(20).std()
    df["Denoised_Close"] = denoise_wavelet(df["Close"].ffill())

    if USE_REGIME:
        df = add_regime(df)

    if USE_SENTIMENT and len(df):
        headline = f"{df['Symbol'].iloc[0]} is expected to perform well in the market."
        try:
            score = score_sentiment([headline])[0]
        except Exception as e:
            logging.warning(f"Sentiment scoring failed for {df['Symbol'].iloc[0]}: {e}")
            score = 0.0
        df["SentimentScore"] = float(score)
    else:
        df["SentimentScore"] = 0.0

    # mock option greeks placeholders
    df["Delta"] = df["Close"].pct_change(1).fillna(0)
    df["Gamma"] = df["Delta"].diff().fillna(0)

    # Drop NaNs created by rolling indicators
    df = df.dropna().reset_index(drop=True)

    # Keep Symbol last (optional)
    cols = [c for c in df.columns if c != "Symbol"] + ["Symbol"]
    return df[cols]

def relabel(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df["Return"] = (
        df.groupby("Symbol")["Close"]
          .shift(-FWD_HORIZON)
          .sub(df["Close"])
          .div(df["Close"])
    )
    df["Target"] = np.select(
        [df["Return"] > UP_THR, df["Return"] < DN_THR],
        [1, -1],
        default=0,
    )
    return df

def summarize(df: pd.DataFrame):
    print("Combined dataset shape:", df.shape)
    print("Range:", df["Datetime"].min(), "-", df["Datetime"].max())
    print("Per-ticker counts:")
    print(df["Symbol"].value_counts().to_string())

    na_cols = df.columns[df.isna().any()]
    if len(na_cols):
        print("\nColumns with NaNs:")
        print(df[na_cols].isna().sum().sort_values(ascending=False).to_string())
    else:
        print("\nNo NaNs detected.")

    print("\nLabel counts:")
    print(df["Target"].value_counts().sort_index().to_string())
    print("\nLabel ratios (%):")
    print((df["Target"].value_counts(normalize=True) * 100).round(2).to_string())

# ============================================================
# BUILD DATASET
# ============================================================
all_dfs: list[pd.DataFrame] = []

for i, ticker in enumerate(SYMBOLS, 1):
    logging.info(f"[{i}/{len(SYMBOLS)}] Processing {ticker}")

    raw = download_stock_data(
        ticker,
        interval=INTERVAL,
        period_days=PERIOD_DAYS,
        max_retries=5,
        sleep_base=3
    )

    if raw is None or raw.empty:
        logging.warning(f"No data for {ticker}")
        continue

    #  Data provenance right after download
    MANIFEST.setdefault("data_provenance", {})
    MANIFEST["data_provenance"][ticker] = {
        "vendor": DATA_VENDOR,
        "interval": INTERVAL,         #  interval recorded here
        "period_days": PERIOD_DAYS,   #  period recorded here
        "rows": int(len(raw)),
        "start": str(raw["Datetime"].min()) if "Datetime" in raw.columns else None,
        "end": str(raw["Datetime"].max()) if "Datetime" in raw.columns else None,
    }
    _safe_json_dump(MANIFEST, MANIFEST_PATH)

    try:
        features = compute_enhanced_features(raw)
        if features is not None and not features.empty:
            logging.info(f"[{ticker}] feature rows: {len(features)}")
            all_dfs.append(features)
        else:
            logging.warning(f"Feature set empty for {ticker}")
    except Exception as e:
        logging.error(f"Feature engineering failed for {ticker}: {e}")
    finally:
        del raw
        try:
            del features
        except NameError:
            pass
        gc.collect()
        time.sleep(0.5)

if not all_dfs:
    logging.warning("No usable data found for any ticker.")
    raise SystemExit

final_df = pd.concat(all_dfs, ignore_index=True)
logging.info(f"Combined dataset shape: {final_df.shape}")

# Drop junk columns if present
if "Repaired?" in final_df.columns:
    final_df = final_df.drop(columns=["Repaired?"])

# ============================================================
# TIMEZONE HANDLING + RTH FILTER + TZ-NAIVE OUTPUT
# ============================================================
dt = pd.to_datetime(final_df["Datetime"], errors="coerce")
if getattr(dt.dt, "tz", None) is None:
    dt = dt.dt.tz_localize("UTC")
else:
    dt = dt.dt.tz_convert("UTC")

# Convert to NY time for RTH filter
final_df["Datetime"] = dt.dt.tz_convert(TIMEZONE_STR)

# Regular trading hours mask (09:30–16:00 ET, weekdays)
dt2 = final_df["Datetime"]
rth_mask = (
    (dt2.dt.weekday < 5) &
    (dt2.dt.time >= pd.to_datetime("09:30").time()) &
    (dt2.dt.time <  pd.to_datetime("16:00").time())
)
final_df = final_df[rth_mask].reset_index(drop=True)

# Strip tz so training script doesn’t choke on tz-aware timestamps
final_df["Datetime"] = final_df["Datetime"].dt.tz_localize(None)

# ============================================================
# LABELING + HORIZON TRIM
# ============================================================
final_df = relabel(final_df)
final_df = final_df.sort_values(["Symbol", "Datetime"]).reset_index(drop=True)

# Create __i/__n then drop last horizon rows per symbol
final_df["__i"] = final_df.groupby("Symbol").cumcount()
final_df["__n"] = final_df.groupby("Symbol")["__i"].transform("max") + 1
final_df = final_df[final_df["__i"] < (final_df["__n"] - FWD_HORIZON)].copy()
final_df = final_df.drop(columns=["__i", "__n"]).reset_index(drop=True)

# ============================================================
# FEATURE MANIFEST (ordered) + NA CLEAN
# ============================================================
exclude = {"Target", "Return", "Symbol", "Datetime"}
feature_cols = [c for c in final_df.columns if c not in exclude]

#  stable/ordered feature manifest
feature_cols = sorted([str(c) for c in feature_cols])
MANIFEST["feature_manifest"] = feature_cols
_safe_json_dump(MANIFEST, MANIFEST_PATH)

final_df = final_df.dropna(subset=feature_cols).reset_index(drop=True)

# Put Target/Return/Symbol at end (matches your training expectations)
ORDER_LAST = ["Target", "Return", "Symbol"]
cols = [c for c in final_df.columns if c not in ORDER_LAST] + ORDER_LAST
final_df = final_df[cols]

# ============================================================
# SUMMARY PRINT
# ============================================================
summarize(final_df)

# ============================================================
# PER-SYMBOL TIME SPLIT (train/val)
# ============================================================
train_parts, val_parts = [], []

for sym, g in final_df.groupby("Symbol", sort=False):
    g = g.sort_values("Datetime").reset_index(drop=True)
    cut = int((1.0 - VAL_FRACTION) * len(g))

    if cut <= 0 or cut >= len(g):
        logging.warning(f"Skipping {sym} due to insufficient rows for split.")
        continue

    train_parts.append(g.iloc[:cut])
    val_parts.append(g.iloc[cut:])

train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)

train_df = train_df.sort_values(["Symbol", "Datetime"]).reset_index(drop=True)
val_df   = val_df.sort_values(["Symbol", "Datetime"]).reset_index(drop=True)

print(f"\nTrain: {train_df.shape},  Val: {val_df.shape}")

# ============================================================
# SAVE LOCALLY (CSV + Parquet) + DRIVE (CSV)
# ============================================================
final_df.to_csv(LOCAL_OUT, index=False)
train_df.to_csv(LOCAL_TRAIN, index=False)
val_df.to_csv(LOCAL_VAL, index=False)

manifest_record("local_full_csv",  os.path.abspath(LOCAL_OUT))
manifest_record("local_train_csv", os.path.abspath(LOCAL_TRAIN))
manifest_record("local_val_csv",   os.path.abspath(LOCAL_VAL))

logging.info(f"Saved local CSVs: {LOCAL_OUT}, {LOCAL_TRAIN}, {LOCAL_VAL}")

final_df.to_parquet(PARQ_FULL, index=False)
train_df.to_parquet(PARQ_TRAIN, index=False)
val_df.to_parquet(PARQ_VAL, index=False)

manifest_record("local_full_parquet",  os.path.abspath(PARQ_FULL))
manifest_record("local_train_parquet", os.path.abspath(PARQ_TRAIN))
manifest_record("local_val_parquet",   os.path.abspath(PARQ_VAL))

logging.info(f"Saved local Parquets: {PARQ_FULL}, {PARQ_TRAIN}, {PARQ_VAL}")

# Save to Google Drive (CSV)
final_path = os.path.join(DRIVE_DIR, LOCAL_OUT)
train_path = os.path.join(DRIVE_DIR, LOCAL_TRAIN)
val_path   = os.path.join(DRIVE_DIR, LOCAL_VAL)

final_df.to_csv(final_path, index=False)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

manifest_record("drive_full_csv",  final_path)
manifest_record("drive_train_csv", train_path)
manifest_record("drive_val_csv",   val_path)

logging.info(f"Saved to Google Drive:\n- {final_path}\n- {train_path}\n- {val_path}")

# ============================================================
# CLEANUP
# ============================================================
del all_dfs, final_df, train_df, val_df
gc.collect()

2026-03-07 22:31:18,949 - INFO - yfinance version: 1.2.0
2026-03-07 22:31:18,950 - INFO - pandas version: 2.2.2
2026-03-07 22:31:18,978 - INFO - [1/3] Processing AAPL
2026-03-07 22:31:18,980 - INFO - [AAPL] Attempt 1: download(period=720d, interval=1h)
2026-03-07 22:31:19,838 - INFO - AAPL: 1h: div-adjust-repair-bad: Repaired missing div-adjust: ['2023-05-12', '2023-08-11', '2023-11-10', '2024-02-09', '2024-05-10', '2024-08-12', '2024-11-08', '2025-02-10', '2025-05-12', '2025-08-11', '2025-11-10', '2026-02-09']
2026-03-07 22:31:20,475 - INFO - [AAPL] rows: 4999 from 2023-04-24 13:30:00 to 2026-03-06 20:30:00
2026-03-07 22:31:20,691 - INFO - [AAPL] feature rows: 4961
2026-03-07 22:31:22,188 - INFO - [2/3] Processing NVDA
2026-03-07 22:31:22,194 - INFO - [NVDA] Attempt 1: download(period=720d, interval=1h)
2026-03-07 22:31:24,312 - INFO - NVDA: 1h: div-adjust-repair-bad: Repaired missing div-adjust: ['2023-06-07', '2023-09-06', '2023-12-05', '2024-03-05', '2024-06-11', '2024-09-12', '202

Combined dataset shape: (14862, 34)
Range: 2023-05-01 12:30:00 - 2026-03-05 12:30:00
Per-ticker counts:
Symbol
MSFT    4960
AAPL    4951
NVDA    4951

No NaNs detected.

Label counts:
Target
-1     1984
 0    10425
 1     2453

Label ratios (%):
Target
 0    70.15
 1    16.51
-1    13.35

Train: (11888, 34),  Val: (2974, 34)


2026-03-07 22:31:33,441 - INFO - Saved local CSVs: multi_stock_feature_engineered_dataset.csv, train.csv, val.csv
2026-03-07 22:31:33,814 - INFO - Saved local Parquets: features_full.parquet, train.parquet, val.parquet
2026-03-07 22:31:38,528 - INFO - Saved to Google Drive:
- /content/drive/MyDrive/trading_data/multi_stock_feature_engineered_dataset.csv
- /content/drive/MyDrive/trading_data/train.csv
- /content/drive/MyDrive/trading_data/val.csv


93

In [9]:
df = pd.read_csv("multi_stock_feature_engineered_dataset.csv")
print(df.head())


              Datetime   Adj Close       Close        High         Low  \
0  2023-05-01 12:30:00  167.710911  170.139999  170.449997  169.985001   
1  2023-05-01 13:30:00  167.493961  169.919907  170.410004  169.820007   
2  2023-05-01 14:30:00  167.268317  169.690994  169.960007  169.199997   
3  2023-05-01 15:30:00  167.139190  169.559998  169.899994  169.250000   
4  2023-05-02 09:30:00  166.084474  168.490005  170.350006  168.485001   

         Open    Volume      SMA_20    STD_20  Upper_Band  ...  \
0  169.985001   5044388  167.958669  1.829613  171.617895  ...   
1  170.130005   3838143  168.273164  1.568324  171.409812  ...   
2  169.899994   5042479  168.571214  1.177114  170.925442  ...   
3  169.699997   5100536  168.736464  1.061108  170.858680  ...   
4  170.089996  11147933  168.826469  0.948776  170.724021  ...   

   Denoised_Close     Vol20     Ret20  Regime4  SentimentScore     Delta  \
0      169.791229  0.004874  0.038959        3             0.0  0.000971   
1     

In [10]:
from __future__ import annotations

# -------------------------
# Imports
# -------------------------
import os
import gc
import time
import json
import logging
import glob
import shutil
import heapq
import random
import platform
import hashlib
import csv
from threading import Lock
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import gymnasium as gym
from gymnasium.spaces import Box as GBox

import yfinance as yf
from gym_anytrading.envs import StocksEnv

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.utils import set_random_seed

import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score


# -------------------------
# Basic Logging
# -------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)


# -------------------------
# Repro
# -------------------------
SEED = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    try:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

    try:
        set_random_seed(seed)
    except Exception:
        pass


seed_everything(SEED)


def now_utc() -> datetime:
    return datetime.now(timezone.utc)


# -------------------------
# Annualization heuristic (intraday: 6.5 hrs * 252)
# -------------------------
def _annualization_factor() -> float:
    return float(np.sqrt(252 * 6.5))


# -------------------------
# Feature engineering import (fallback)
# -------------------------
FEATURES_FALLBACK_ACTIVE = False
try:
    from feature_engineering import compute_enhanced_features  # your real pipeline if present
except Exception as e:
    FEATURES_FALLBACK_ACTIVE = True
    logging.warning(f"[WARN] feature_engineering import failed; fallback active. Err: {e}")

    def compute_enhanced_features(df_in: pd.DataFrame) -> pd.DataFrame:
        df = df_in.copy()
        if "Datetime" in df.columns:
            df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
            df = df.sort_values("Datetime").reset_index(drop=True)
        if "Close" not in df.columns:
            raise ValueError("compute_enhanced_features fallback: missing 'Close'")
        return df


# -------------------------
# Flags / knobs
# -------------------------
ENABLE_VOL_CONTROLS = True
VOL_LOOKBACK = 20
VOL_EWM_SPAN = 20
VOL_GATE_Q = 0.995
VOL_SCALE_K = 0.75
VOL_MIN_SCALE = 0.65

TP_BASE = 0.045
SL_BASE = 0.030
TP_MIN, TP_MAX = 0.015, 0.100
SL_MIN, SL_MAX = 0.010, 0.080

ENABLE_SENTIMENT = False
ENABLE_DRAWDOWN_CIRCUIT_BREAKER = True
ENABLE_WAVELET = True

TEST_MODE = True
ENABLE_PLOTS = False
LIVE_MODE = False
SIM_LATENCY_MS = 0
BROKER = "log"
MAX_WORKERS = 1
FORCE_RETRAIN = False  # set True when you want to ignore existing artifacts and retrain

# -------------------------
# Phase 2 Gate Config
# -------------------------
ENABLE_TRADE_GATE = True
GATE_MODEL_KIND = "rf"              # "rf" for now
GATE_LABEL_HORIZON = 5              # future bars for supervised label
GATE_LABEL_RETURN_THR = 0.003       # 0.30% future return threshold for success label
GATE_THRESHOLD = 0.55               # pass gate only if p_win > threshold
GATE_MIN_TRAIN_ROWS = 200
GATE_TRAIN_FRACTION = 0.70
GATE_TOP_EXPORT_TO_QC = True

# Back-compat alias
test_mode = TEST_MODE

if LIVE_MODE and FEATURES_FALLBACK_ACTIVE:
    raise RuntimeError("LIVE_MODE=True but compute_enhanced_features fallback active. Fix import first.")


# -------------------------
# Paths (Colab or local)
# -------------------------
def _detect_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


IN_COLAB = _detect_colab()

try:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_BASE = "/content/drive/MyDrive"
    else:
        DRIVE_BASE = os.getcwd()
except Exception:
    DRIVE_BASE = os.getcwd()

BASE_RESULTS_DIR = os.path.join(DRIVE_BASE, "Results_May_2025")
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_RESULTS_DIR = os.path.join(BASE_RESULTS_DIR, f"ppo_walkforward_results_{RUN_TAG}")
FINAL_MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "ppo_models_master")
QC_TOP_DIR = os.path.join(BASE_RESULTS_DIR, "ppo_models_QC_TOP")

os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
os.makedirs(QC_TOP_DIR, exist_ok=True)


# -------------------------
# Force-retrain cleanup
# -------------------------
def safe_reset_dir(dir_path: str):
    """
    Remove and recreate a directory safely.
    Only use on directories that belong to this experiment pipeline.
    """
    if os.path.isdir(dir_path):
        shutil.rmtree(dir_path)
    os.makedirs(dir_path, exist_ok=True)


def cleanup_prefix_artifacts(base_dir: str, prefix: str):
    """
    Delete all files for one model/window prefix inside a known pipeline directory.
    Example prefix: ppo_AAPL_window1
    """
    if not os.path.isdir(base_dir):
        return

    patterns = [
        f"{prefix}_model.zip",
        f"{prefix}_vecnorm.pkl",
        f"{prefix}_features.json",
        f"{prefix}_probability_config.json",
        f"{prefix}_model_info.json",
        f"{prefix}_predictions.csv",
        f"{prefix}_predictions_compat.csv",
        f"{prefix}_trades.csv",
        f"{prefix}_step_log.csv",
        f"{prefix}_eval.csv",
        f"{prefix}_metrics.json",

        # gate artifacts
        f"{prefix}_gate_model.pkl",
        f"{prefix}_gate_features.json",
        f"{prefix}_gate_config.json",
    ]

    for pat in patterns:
        for fp in glob.glob(os.path.join(base_dir, pat)):
            try:
                if os.path.isfile(fp):
                    os.remove(fp)
            except Exception as e:
                logging.warning(f"[FORCE_RETRAIN] Could not delete {fp}: {e}")


def cleanup_for_force_retrain(
    run_results_dir: str,
    final_model_dir: str,
    qc_top_dir: str,
    tickers: list[str] | None = None,
    max_windows: int = 50,
):
    logging.info("[FORCE_RETRAIN] Starting artifact cleanup...")

    safe_reset_dir(run_results_dir)

    if tickers:
        for ticker in tickers:
            for w_idx in range(1, max_windows + 1):
                prefix = f"ppo_{ticker}_window{w_idx}"
                cleanup_prefix_artifacts(final_model_dir, prefix)
                cleanup_prefix_artifacts(qc_top_dir, prefix)

    logging.info("[FORCE_RETRAIN] Artifact cleanup complete.")


SELECTOR_FULL_PATH = os.path.join(BASE_RESULTS_DIR, "ppo_model_selector_FULL.csv")
SELECTOR_JSON_PATH = os.path.join(BASE_RESULTS_DIR, "ppo_model_selector_final.json")

# Global skip log
SKIP_AGG_PATH = os.path.join(RUN_RESULTS_DIR, "skipped_windows_global.csv")
SKIP_LOCK = Lock()


# -------------------------
# Device gate
# -------------------------
USE_GPU = torch.cuda.is_available() and (MAX_WORKERS == 1)
DEVICE_STR = "cuda" if USE_GPU else "cpu"
logging.info(
    f"[CONFIG] IN_COLAB={IN_COLAB} | MAX_WORKERS={MAX_WORKERS} | "
    f"cuda={torch.cuda.is_available()} | TEST_MODE={TEST_MODE} | DEVICE={DEVICE_STR}"
)


# -------------------------
# Run-level artifact manifest
# -------------------------
ENABLE_HASHES = False  # set True if you want sha256; slower
ARTIFACT_MANIFEST_PATH = os.path.join(RUN_RESULTS_DIR, "artifact_manifest.json")


def _sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _safe_json_dump(obj, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def load_manifest() -> dict:
    if os.path.exists(ARTIFACT_MANIFEST_PATH):
        try:
            with open(ARTIFACT_MANIFEST_PATH, "r") as f:
                return json.load(f)
        except Exception:
            pass

    return {
        "run_tag": RUN_TAG,
        "created_utc": now_utc().isoformat(),
        "run_results_dir": RUN_RESULTS_DIR,
        "final_model_dir": FINAL_MODEL_DIR,
        "qc_top_dir": QC_TOP_DIR,
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "torch": getattr(torch, "__version__", None),
            "gymnasium": getattr(gym, "__version__", None),
            "stable_baselines3": None,
            "pandas": getattr(pd, "__version__", None),
            "numpy": getattr(np, "__version__", None),
            "yfinance": getattr(yf, "__version__", None),
        },
        "artifacts": {}
    }


MANIFEST = load_manifest()
try:
    import stable_baselines3
    MANIFEST["environment"]["stable_baselines3"] = stable_baselines3.__version__
except Exception:
    pass
_safe_json_dump(MANIFEST, ARTIFACT_MANIFEST_PATH)


def manifest_update(prefix: str, updates: dict):
    MANIFEST["artifacts"].setdefault(prefix, {})
    MANIFEST["artifacts"][prefix].update(updates)
    MANIFEST["artifacts"][prefix]["updated_utc"] = now_utc().isoformat()
    _safe_json_dump(MANIFEST, ARTIFACT_MANIFEST_PATH)


def manifest_record_paths(prefix: str, paths: dict, do_hashes: bool = False):
    rec = {}
    for k, p in paths.items():
        rec[k] = {"path": p, "exists": bool(p and os.path.exists(p))}
        if do_hashes and p and os.path.exists(p):
            try:
                rec[k]["sha256"] = _sha256_file(p)
            except Exception as e:
                rec[k]["sha256_error"] = str(e)
    manifest_update(prefix, {"paths": rec})


def write_master_bundle(
    prefix: str,
    features: list[str],
    threshold: float,
    model_info: dict,
    gate_features: list[str] | None = None,
    gate_threshold: float | None = None,
):
    f_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_features.json")
    p_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_probability_config.json")
    i_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_model_info.json")

    _safe_json_dump({"features": list(features)}, f_path)
    _safe_json_dump(
        {
            "threshold": float(threshold),
            "use_confidence": True,
            "inference_mode": "deterministic",
        },
        p_path
    )
    _safe_json_dump(model_info, i_path)

    record_paths = {
        "master_features_json": f_path,
        "master_probability_config_json": p_path,
        "master_model_info_json": i_path,
    }

    if gate_features is not None:
        gf_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_features.json")
        gc_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_config.json")
        _safe_json_dump({"features": list(gate_features)}, gf_path)
        _safe_json_dump(
            {
                "gate_enabled": True,
                "gate_model_type": GATE_MODEL_KIND,
                "threshold": float(gate_threshold if gate_threshold is not None else GATE_THRESHOLD),
                "label_horizon": int(GATE_LABEL_HORIZON),
                "label_return_threshold": float(GATE_LABEL_RETURN_THR),
                "uses_scaler": False,
            },
            gc_path,
        )
        record_paths["master_gate_features_json"] = gf_path
        record_paths["master_gate_config_json"] = gc_path

    manifest_record_paths(prefix, record_paths, do_hashes=ENABLE_HASHES)


def save_gate_model(prefix: str, gate_model, gate_features: list[str], gate_config: dict):
    gm_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_model.pkl")
    gf_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_features.json")
    gc_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_config.json")

    joblib.dump(gate_model, gm_path)
    _safe_json_dump({"features": list(gate_features)}, gf_path)
    _safe_json_dump(gate_config, gc_path)

    manifest_record_paths(
        prefix,
        {
            "gate_model_pkl": gm_path,
            "gate_features_json": gf_path,
            "gate_config_json": gc_path,
        },
        do_hashes=ENABLE_HASHES,
    )
    return gm_path, gf_path, gc_path


# -------------------------
# Health check helper
# -------------------------
def health_check_latest_run(base_dir: str) -> dict:
    run_dirs = sorted(glob.glob(os.path.join(base_dir, "ppo_walkforward_results_*")))[::-1]
    if not run_dirs:
        return {"ok": False, "reason": "No run folders found."}

    latest = run_dirs[0]
    summaries = sorted(glob.glob(os.path.join(latest, "summary*.csv")))
    out = {
        "ok": True,
        "latest_run_dir": latest,
        "summary_files": summaries,
        "rows_total": 0,
        "tickers": [],
    }

    frames = []
    for p in summaries:
        try:
            frames.append(pd.read_csv(p))
        except Exception:
            pass

    if frames:
        combo = pd.concat(frames, ignore_index=True)
        out["rows_total"] = int(len(combo))
        if "Ticker" in combo.columns:
            out["tickers"] = sorted(list(combo["Ticker"].dropna().astype(str).unique()))
    else:
        out["ok"] = False
        out["reason"] = "Latest run has no readable summary*.csv"

    return out


# -------------------------
# Data load (training uses prebuilt CSV)
# -------------------------
DATA_CANDIDATES = [
    "multi_stock_feature_engineered_dataset.csv",
    os.path.join(DRIVE_BASE, "trading_data", "multi_stock_feature_engineered_dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
if not DATA_PATH:
    raise FileNotFoundError(
        "Could not find multi_stock_feature_engineered_dataset.csv in cwd or Drive/trading_data."
    )

df = pd.read_csv(DATA_PATH)

if "Datetime" not in df.columns:
    raise ValueError("Dataset missing 'Datetime' column.")
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.dropna(subset=["Datetime"]).reset_index(drop=True)

if "Symbol" not in df.columns:
    raise ValueError("Dataset missing 'Symbol' column.")
if "Close" not in df.columns:
    raise ValueError("Dataset missing 'Close' column.")

if ENABLE_WAVELET and "Denoised_Close" not in df.columns:
    logging.warning("ENABLE_WAVELET=True but 'Denoised_Close' missing; copying Close -> Denoised_Close.")
    df["Denoised_Close"] = df["Close"]

# Save run config + provenance
RUN_CONFIG_PATH = os.path.join(RUN_RESULTS_DIR, "config.json")
FEATURE_MANIFEST_PATH = os.path.join(RUN_RESULTS_DIR, "feature_manifest.json")
DATA_PROVENANCE_PATH = os.path.join(RUN_RESULTS_DIR, "data_provenance.json")


def build_ordered_feature_list(df_in: pd.DataFrame) -> list[str]:
    exclude = {"Datetime", "Symbol", "Target", "Return"}
    feats = [
        str(c) for c in df_in.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df_in[c])
    ]
    return sorted(feats)


def build_data_provenance_from_df(df_in: pd.DataFrame) -> dict:
    prov = {
        "source": "prebuilt_csv",
        "data_path": os.path.abspath(DATA_PATH),
        "data_sha256": _sha256_file(DATA_PATH) if ENABLE_HASHES else None,
        "rows_total": int(len(df_in)),
        "symbols": sorted(df_in["Symbol"].dropna().astype(str).unique().tolist()),
        "per_symbol": {},
    }
    for sym, g in df_in.groupby("Symbol", sort=False):
        g2 = g.sort_values("Datetime")
        prov["per_symbol"][str(sym)] = {
            "rows": int(len(g2)),
            "start": str(g2["Datetime"].iloc[0]),
            "end": str(g2["Datetime"].iloc[-1]),
        }
    return prov


RUN_CONFIG = {
    "run_tag": RUN_TAG,
    "created_utc": now_utc().isoformat(),
    "paths": {
        "run_results_dir": RUN_RESULTS_DIR,
        "final_model_dir": FINAL_MODEL_DIR,
        "qc_top_dir": QC_TOP_DIR,
        "base_results_dir": BASE_RESULTS_DIR,
        "dataset_csv": os.path.abspath(DATA_PATH),
    },
    "seed": SEED,
    "device": DEVICE_STR,
    "flags": {
        "ENABLE_VOL_CONTROLS": ENABLE_VOL_CONTROLS,
        "ENABLE_SENTIMENT": ENABLE_SENTIMENT,
        "ENABLE_WAVELET": ENABLE_WAVELET,
        "ENABLE_DRAWDOWN_CIRCUIT_BREAKER": ENABLE_DRAWDOWN_CIRCUIT_BREAKER,
        "LIVE_MODE": LIVE_MODE,
        "TEST_MODE": TEST_MODE,
        "MAX_WORKERS": MAX_WORKERS,
        "FORCE_RETRAIN": FORCE_RETRAIN,
        "FEATURES_FALLBACK_ACTIVE": FEATURES_FALLBACK_ACTIVE,
        "ENABLE_TRADE_GATE": ENABLE_TRADE_GATE,
    },
    "feature_pipeline": {
        "fallback_active": FEATURES_FALLBACK_ACTIVE,
        "module_import_ok": not FEATURES_FALLBACK_ACTIVE,
    },
    "gate_config": {
        "enabled": ENABLE_TRADE_GATE,
        "model_kind": GATE_MODEL_KIND,
        "label_horizon": GATE_LABEL_HORIZON,
        "label_return_threshold": GATE_LABEL_RETURN_THR,
        "threshold": GATE_THRESHOLD,
    },
}
_safe_json_dump(RUN_CONFIG, RUN_CONFIG_PATH)

ORDERED_FEATURES = build_ordered_feature_list(df)
_safe_json_dump({"features_ordered": ORDERED_FEATURES}, FEATURE_MANIFEST_PATH)

PROVENANCE = build_data_provenance_from_df(df)
_safe_json_dump(PROVENANCE, DATA_PROVENANCE_PATH)

manifest_record_paths(
    "RUN_LEVEL",
    {
        "config_json": RUN_CONFIG_PATH,
        "feature_manifest_json": FEATURE_MANIFEST_PATH,
        "data_provenance_json": DATA_PROVENANCE_PATH,
        "dataset_csv": os.path.abspath(DATA_PATH),
    },
    do_hashes=ENABLE_HASHES,
)


# -------------------------
# Skip logging
# -------------------------
def record_skips_global(
    ticker: str,
    skipped_windows: list[str],
    total_windows: int | None = None,
    fully_skipped: bool = False,
):
    if not skipped_windows and not fully_skipped:
        return

    with SKIP_LOCK:
        new_file = not os.path.exists(SKIP_AGG_PATH)
        with open(SKIP_AGG_PATH, "a", newline="") as f:
            w = csv.writer(f)
            if new_file:
                w.writerow(["Ticker", "Window", "FullySkipped", "TotalWindows"])

            if fully_skipped:
                w.writerow([ticker, "ALL", True, total_windows if total_windows is not None else ""])
            else:
                for wname in skipped_windows:
                    win = ""
                    try:
                        _, win_str = wname.split("_window")
                        win = int(win_str)
                    except Exception:
                        pass
                    w.writerow([ticker, win, False, total_windows if total_windows is not None else ""])


# -------------------------
# Env kwargs
# -------------------------
ENV_KWARGS = dict(
    window_size=10,
    cost_rate=0.0002,
    slip_rate=0.0003,
    k_alpha=0.0,
    k_mom=0.15,
    k_sent=(0.01 if ENABLE_SENTIMENT else 0.0),
    mom_source="denoised",
    mom_lookback=20,
    min_trade_delta=0.06,
    cooldown=8,
    reward_clip=0.05,
    k_vol=0.00,
    k_dd=0.00,
    enable_slo=ENABLE_DRAWDOWN_CIRCUIT_BREAKER,
    slo_max_dd=0.15,
    slo_lock_steps=15,
    slo_hard_stop_enabled=False,
    slo_hard_stop_pct=0.01,
    slo_chop_enabled=True,
    slo_chop_col="Chop",
    slo_chop_lock_steps=3,
    enable_vol_controls=ENABLE_VOL_CONTROLS,
    vol_col="Vol_EWM",
    vol_gate_quantile=VOL_GATE_Q,
    vol_scale_k=VOL_SCALE_K,
    vol_min_scale=VOL_MIN_SCALE,
    tp_base=TP_BASE,
    sl_base=SL_BASE,
    tp_min=TP_MIN,
    tp_max=TP_MAX,
    sl_min=SL_MIN,
    sl_max=SL_MAX,
)


# -------------------------
# ContinuousPositionEnv
# -------------------------
class ContinuousPositionEnv(StocksEnv):
    def __init__(self, df: pd.DataFrame, frame_bound: tuple[int, int], **kwargs):
        if "window_size" not in kwargs:
            raise ValueError("ContinuousPositionEnv requires window_size in kwargs.")
        window_size = int(kwargs.pop("window_size"))

        self.cost_rate = float(kwargs.pop("cost_rate", 0.0002))
        self.slip_rate = float(kwargs.pop("slip_rate", 0.0003))
        self.k_alpha = float(kwargs.pop("k_alpha", 0.0))
        self.k_mom = float(kwargs.pop("k_mom", 0.15))
        self.k_sent = float(kwargs.pop("k_sent", 0.0))
        self.mom_source = str(kwargs.pop("mom_source", "denoised"))
        self.mom_lookback = int(kwargs.pop("mom_lookback", 20))
        self.min_trade_delta = float(kwargs.pop("min_trade_delta", 0.04))
        self.cooldown = int(kwargs.pop("cooldown", 6))
        self.reward_clip = float(kwargs.pop("reward_clip", 0.05))
        self.k_vol = float(kwargs.pop("k_vol", 0.0))
        self.k_dd = float(kwargs.pop("k_dd", 0.0))

        self.enable_vol_controls = bool(kwargs.pop("enable_vol_controls", False))
        self.vol_col = str(kwargs.pop("vol_col", "Vol_EWM"))
        self.vol_gate_quantile = float(kwargs.pop("vol_gate_quantile", 0.95))
        self.vol_scale_k = float(kwargs.pop("vol_scale_k", 2.0))
        self.vol_min_scale = float(kwargs.pop("vol_min_scale", 0.25))

        self.tp_base = float(kwargs.pop("tp_base", 0.03))
        self.sl_base = float(kwargs.pop("sl_base", 0.02))
        self.tp_min = float(kwargs.pop("tp_min", 0.01))
        self.tp_max = float(kwargs.pop("tp_max", 0.08))
        self.sl_min = float(kwargs.pop("sl_min", 0.005))
        self.sl_max = float(kwargs.pop("sl_max", 0.06))

        self.enable_slo = bool(kwargs.pop("enable_slo", False))
        self.slo_max_dd = float(kwargs.pop("slo_max_dd", 0.15))
        self.slo_lock_steps = int(kwargs.pop("slo_lock_steps", 15))
        self.slo_hard_stop_enabled = bool(kwargs.pop("slo_hard_stop_enabled", False))
        self.slo_hard_stop_pct = float(kwargs.pop("slo_hard_stop_pct", 0.01))

        self.slo_chop_enabled = bool(kwargs.pop("slo_chop_enabled", False))
        self.slo_chop_col = str(kwargs.pop("slo_chop_col", "Chop"))
        self.slo_chop_lock_steps = int(kwargs.pop("slo_chop_lock_steps", 5))
        self._chop_lock_until = -1

        if kwargs:
            raise ValueError(f"Unexpected env kwargs: {list(kwargs.keys())}")

        super().__init__(df=df.reset_index(drop=True), frame_bound=frame_bound, window_size=window_size)

        self._parent_action_space = self.action_space

        if isinstance(self.observation_space, gym.spaces.Box):
            self.observation_space = GBox(
                low=self.observation_space.low,
                high=self.observation_space.high,
                shape=self.observation_space.shape,
                dtype=self.observation_space.dtype,
            )

        self.nav = 1.0
        self.pos = 0.0
        self.peak_nav = 1.0
        self.nav_history = []
        self.ret_history = []
        self.trade_count = 0
        self._last_trade_step = -self.cooldown
        self._slo_lock_until = -1

        self.entry_price = None
        self.entry_tick = None

        self.action_space = GBox(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

    def reset(self, **kwargs):
        out = super().reset(**kwargs)
        if isinstance(out, tuple) and len(out) == 2:
            obs, info = out
        else:
            obs, info = out, {}

        self.nav = 1.0
        self.pos = 0.0
        self.peak_nav = 1.0
        self.nav_history = [self.nav]
        self.ret_history = []
        self.trade_count = 0
        self._last_trade_step = -self.cooldown
        self._slo_lock_until = -1
        self.entry_price = None
        self.entry_tick = None
        self._chop_lock_until = -1

        self._vol_ref = float(self.df["Vol_Ref"].iloc[0]) if "Vol_Ref" in self.df.columns else 0.0
        self._vol_gate = float(self.df["Vol_Gate"].iloc[0]) if "Vol_Gate" in self.df.columns else 0.0

        info = info or {}
        info.update({"nav": self.nav, "pos": self.pos, "trade_count": int(self.trade_count)})
        return obs, info

    def _ret_t(self) -> float:
        cur = float(self.df.loc[self._current_tick, "Close"])
        prev = float(self.df.loc[max(self._current_tick - 1, 0), "Close"])
        return 0.0 if prev <= 0 else (cur - prev) / prev

    def _price_t(self) -> float:
        return float(self.df.loc[self._current_tick, "Close"])

    def _vol_t(self) -> float:
        if not self.enable_vol_controls:
            return 0.0
        if self.vol_col in self.df.columns:
            try:
                return float(self.df.loc[self._current_tick, self.vol_col])
            except Exception:
                return 0.0
        return 0.0

    def _mom_signal(self) -> float:
        if self.mom_source == "macd" and "MACD_Line" in self.df.columns:
            recent = self.df["MACD_Line"].iloc[max(self._current_tick - 200, 0):self._current_tick + 1]
            denom = 1e-6 + float(recent.std())
            return float(np.tanh(float(self.df.loc[self._current_tick, "MACD_Line"]) / denom))

        if "Denoised_Close" in self.df.columns and self._current_tick - self.mom_lookback >= 0:
            now = float(self.df.loc[self._current_tick, "Denoised_Close"])
            then = float(self.df.loc[self._current_tick - self.mom_lookback, "Denoised_Close"])
            base = float(self.df.loc[max(self._current_tick - 1, 0), "Close"])
            slope = (now - then) / max(self.mom_lookback, 1)
            return float(np.tanh(10.0 * (slope / max(abs(base), 1e-6))))

        return 0.0

    def _dynamic_tp_sl(self, vol_t: float) -> tuple[float, float]:
        vol_ref = float(getattr(self, "_vol_ref", 0.0))
        eps = 1e-9
        ratio = (vol_ref / max(vol_t, eps)) if vol_ref > 0 else 1.0
        tp = float(np.clip(self.tp_base * ratio, self.tp_min, self.tp_max))
        sl = float(np.clip(self.sl_base * ratio, self.sl_min, self.sl_max))
        return tp, sl

    def _base_ret_next(self) -> float:
        try:
            cur = float(self.df.loc[self._current_tick, "Close"])
            prev = float(self.df.loc[max(self._current_tick - 1, 0), "Close"])
            return 0.0 if prev <= 0 else (cur - prev) / prev
        except Exception:
            return 0.0

    def _step_parent_hold(self):
        cur_as = self.action_space
        try:
            self.action_space = self._parent_action_space
            out = super().step(0)
        finally:
            self.action_space = cur_as

        if len(out) == 5:
            obs, _, terminated, truncated, info = out
            return obs, bool(terminated), bool(truncated), info
        if len(out) == 4:
            obs, _, done, info = out
            return obs, bool(done), False, info
        raise ValueError(f"Unexpected parent step() return length: {len(out)}")

    def step(self, action):
        a = float(np.array(action).squeeze())
        next_pos = float(np.clip(a, -1.0, 1.0))

        forced_close = False
        tp_pct = 0.0
        sl_pct = 0.0
        scale = 1.0
        gated = False
        slo_triggered = False

        r_t = self._ret_t()
        vol_t = self._vol_t()

        if self.enable_vol_controls and (abs(self.pos) > 1e-9) and (self.entry_price is not None):
            tp_pct, sl_pct = self._dynamic_tp_sl(vol_t)
            px = self._price_t()
            pnl_dir = ((px - float(self.entry_price)) / max(float(self.entry_price), 1e-9)) * float(np.sign(self.pos))
            if pnl_dir >= tp_pct or pnl_dir <= -sl_pct:
                next_pos = 0.0
                forced_close = True

        locked = self.enable_slo and (self._current_tick < self._slo_lock_until)
        if locked and not forced_close:
            next_pos = float(self.pos)

        vol_ref = float(getattr(self, "_vol_ref", 0.0))
        vol_gate = float(getattr(self, "_vol_gate", 0.0))

        if self.enable_vol_controls and vol_ref > 0 and vol_t > vol_ref:
            scale = 1.0 / (1.0 + self.vol_scale_k * max(0.0, (vol_t / (vol_ref + 1e-9)) - 1.0))
            scale = float(np.clip(scale, self.vol_min_scale, 1.0))
            next_pos = float(np.clip(next_pos * scale, -1.0, 1.0))

        if self.enable_vol_controls and vol_gate > 0 and vol_t >= vol_gate and not forced_close:
            gated = True
            if abs(self.pos) < 1e-9:
                next_pos = 0.0
            else:
                if np.sign(next_pos) != np.sign(self.pos):
                    next_pos = 0.0
                elif abs(next_pos) > abs(self.pos):
                    next_pos = float(self.pos)

        chop_now = False
        if self.slo_chop_enabled and (self.slo_chop_col in self.df.columns):
            try:
                chop_now = bool(int(self.df.loc[self._current_tick, self.slo_chop_col]) == 1)
            except Exception:
                chop_now = False

        if chop_now:
            self._chop_lock_until = max(self._chop_lock_until, self._current_tick + int(self.slo_chop_lock_steps))
        chop_locked = self._current_tick < self._chop_lock_until

        if chop_locked and not forced_close:
            if abs(self.pos) < 1e-9:
                next_pos = 0.0
            else:
                if np.sign(next_pos) != np.sign(self.pos):
                    next_pos = 0.0
                elif abs(next_pos) > abs(self.pos):
                    next_pos = float(self.pos)

        if forced_close:
            changed = abs(self.pos) > 1e-9
        else:
            changed = (
                (abs(next_pos - self.pos) >= self.min_trade_delta)
                and ((self._current_tick - self._last_trade_step) >= self.cooldown)
            )

        if gated and not forced_close:
            changed = False

        delta_pos = (next_pos - self.pos) if changed else 0.0
        fee_cost = self.cost_rate * abs(delta_pos)
        slip_cost = self.slip_rate * abs(delta_pos)
        trade_cost = fee_cost + slip_cost
        turnover_step = abs(delta_pos)

        pnl_from_prev_pos = self.pos * r_t
        rel_alpha = pnl_from_prev_pos - r_t
        mom_term = self.pos * self._mom_signal()
        shaped = pnl_from_prev_pos + (self.k_mom * mom_term) - trade_cost
        reward = float(np.clip(shaped, -self.reward_clip, self.reward_clip))

        next_nav = self.nav * (1.0 + pnl_from_prev_pos - trade_cost)

        dd_now = 0.0
        if self.enable_slo:
            prev_peak = self.peak_nav
            dd_now = (prev_peak - next_nav) / max(prev_peak, 1e-9)
            if dd_now >= self.slo_max_dd:
                slo_triggered = True
                if abs(self.pos) > 1e-9:
                    trade_cost += (self.cost_rate * abs(self.pos)) + (self.slip_rate * abs(self.pos))
                    next_nav = self.nav * (1.0 + pnl_from_prev_pos - trade_cost)
                self.pos = 0.0
                next_pos = 0.0
                self._slo_lock_until = self._current_tick + int(self.slo_lock_steps)
                self._last_trade_step = self._current_tick
                changed = False
                delta_pos = 0.0
                dd_now = (self.peak_nav - next_nav) / max(self.peak_nav, 1e-9)

        self.nav = float(next_nav)
        self.nav_history.append(self.nav)
        self.peak_nav = max(self.peak_nav, self.nav)

        executed_trade = False
        if changed:
            prev_pos = float(self.pos)
            self.pos = float(next_pos)
            self._last_trade_step = self._current_tick
            self.trade_count += 1
            executed_trade = True

            if abs(self.pos) < 1e-9:
                self.entry_price = None
                self.entry_tick = None
            else:
                if (abs(prev_pos) < 1e-9) or (np.sign(prev_pos) != np.sign(self.pos)):
                    self.entry_price = self._price_t()
                    self.entry_tick = int(self._current_tick)

        obs, terminated, truncated, info = self._step_parent_hold()
        base_ret = float(self._base_ret_next())

        info = info or {}
        info.update({
            "ret_t": float(r_t),
            "base_ret": float(base_ret),
            "fee_cost": float(fee_cost),
            "slip_cost": float(slip_cost),
            "turnover_step": float(turnover_step),
            "delta_pos": float(delta_pos),
            "nav": float(self.nav),
            "pos": float(self.pos),
            "trade_cost": float(trade_cost),
            "pnl_from_prev_pos": float(pnl_from_prev_pos),
            "rel_alpha": float(rel_alpha),
            "mom": float(mom_term),
            "changed": bool(changed),
            "executed_trade": bool(executed_trade),
            "trade_count": int(self.trade_count),
            "drawdown": float(dd_now),
            "slo_triggered": bool(slo_triggered),
            "slo_locked": bool(self.enable_slo and (self._current_tick < self._slo_lock_until)),
            "slo_lock_until": int(self._slo_lock_until),
            "vol_t": float(vol_t),
            "vol_ref": float(vol_ref),
            "vol_gate": float(vol_gate),
            "vol_scale": float(scale),
            "vol_gated": bool(gated),
            "tp_pct": float(tp_pct),
            "sl_pct": float(sl_pct),
            "forced_close": bool(forced_close),
            "chop_now": bool(chop_now),
            "chop_locked": bool(chop_locked),
            "chop_lock_until": int(self._chop_lock_until),
        })
        return obs, reward, terminated, truncated, info


# -------------------------
# PPO policy stats helper
# -------------------------
def get_mu_sigma(model: PPO, obs):
    with torch.no_grad():
        obs_t, _ = model.policy.obs_to_tensor(obs)
        features = model.policy.extract_features(obs_t)
        latent_pi, _ = model.policy.mlp_extractor(features)
        mean_actions = model.policy.action_net(latent_pi)
        mu = float(mean_actions.detach().cpu().numpy().reshape(-1).mean())
        sigma_vec = model.policy.log_std.exp().detach().cpu().numpy().reshape(-1)
        sigma = float(sigma_vec.mean())
    return mu, sigma


# -------------------------
# Windows
# -------------------------
def get_walk_forward_windows(df_in: pd.DataFrame, window_size=3500, step_size=500, min_len=1200):
    return [
        (start, start + window_size)
        for start in range(0, len(df_in) - min_len, step_size)
        if start + window_size <= len(df_in)
    ]


# -------------------------
# Gate Helpers
# -------------------------
def build_trade_gate_label(df_in: pd.DataFrame, horizon: int, ret_thr: float) -> pd.DataFrame:
    df_local = df_in.copy()
    df_local["Gate_Future_Return"] = (
        df_local["Close"].shift(-horizon).sub(df_local["Close"]).div(df_local["Close"])
    )
    df_local["Trade_Label"] = (df_local["Gate_Future_Return"] > ret_thr).astype(int)
    return df_local


def choose_gate_features(df_in: pd.DataFrame) -> list[str]:
    priority = [
        "RSI", "MACD_Line", "MACD_Signal", "ATR", "Volatility",
        "Vol_EWM", "Vol_Roll", "Vol_Ref", "Vol_Gate",
        "Ret_1", "Ret1", "Ret20", "FlipRate20", "Chop",
        "Denoised_Close", "SMA_20", "STD_20", "Upper_Band", "Lower_Band",
        "Stoch", "ROC", "OBV", "CCI", "EMA_10", "EMA_50",
        "Regime4", "SentimentScore", "Delta", "Gamma",
    ]
    feats = [c for c in priority if c in df_in.columns and pd.api.types.is_numeric_dtype(df_in[c])]

    if not feats:
        exclude = {
            "Datetime", "Symbol", "Target", "Return",
            "Trade_Label", "Gate_Future_Return",
        }
        feats = [
            c for c in df_in.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(df_in[c])
        ]
        feats = sorted(feats)

    return feats


def train_trade_gate(
    df_gate: pd.DataFrame,
    gate_features: list[str],
    threshold: float,
) -> dict:
    work = df_gate.copy()
    work = work.dropna(subset=gate_features + ["Trade_Label"]).reset_index(drop=True)

    if len(work) < GATE_MIN_TRAIN_ROWS:
        raise ValueError(f"Gate training skipped: only {len(work)} rows after dropna.")

    split_idx = max(int(len(work) * GATE_TRAIN_FRACTION), 1)
    if split_idx >= len(work):
        split_idx = len(work) - 1

    train_part = work.iloc[:split_idx].copy()
    eval_part = work.iloc[split_idx:].copy()

    X_train = train_part[gate_features].astype(float)
    y_train = train_part["Trade_Label"].astype(int)

    X_eval = eval_part[gate_features].astype(float)
    y_eval = eval_part["Trade_Label"].astype(int)

    if y_train.nunique() < 2:
        raise ValueError("Gate training skipped: training labels have only one class.")

    gate_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    gate_model.fit(X_train, y_train)

    eval_probs = gate_model.predict_proba(X_eval)[:, 1] if len(X_eval) else np.array([])
    eval_pred = (eval_probs > threshold).astype(int) if len(eval_probs) else np.array([])

    auc = np.nan
    precision = np.nan
    recall = np.nan
    if len(eval_probs) and len(np.unique(y_eval)) > 1:
        try:
            auc = float(roc_auc_score(y_eval, eval_probs))
        except Exception:
            pass
    if len(eval_probs):
        try:
            precision = float(precision_score(y_eval, eval_pred, zero_division=0))
        except Exception:
            pass
        try:
            recall = float(recall_score(y_eval, eval_pred, zero_division=0))
        except Exception:
            pass

    return {
        "model": gate_model,
        "features": gate_features,
        "threshold": float(threshold),
        "train_rows": int(len(train_part)),
        "eval_rows": int(len(eval_part)),
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "train_pos_rate": float(y_train.mean()),
        "eval_pos_rate": float(y_eval.mean()) if len(y_eval) else np.nan,
    }


def gate_predict_row(gate_model, feature_row_df: pd.DataFrame) -> float:
    try:
        proba = gate_model.predict_proba(feature_row_df.astype(float))[0, 1]
        return float(proba)
    except Exception:
        return 0.0


# -------------------------
# QC Save
# -------------------------
def save_quantconnect_model(artifact: dict, prefix: str, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)

    model_dst = os.path.join(save_dir, f"{prefix}_model.zip")
    model_obj = artifact.get("model", None)
    model_src = artifact.get("model_path", None)

    try:
        if model_obj is not None:
            model_obj.save(model_dst)
        else:
            if model_src and os.path.exists(model_src):
                if os.path.abspath(model_src) != os.path.abspath(model_dst):
                    shutil.copyfile(model_src, model_dst)
    except Exception as e:
        logging.warning(f"[QC SAVE] model issue for {prefix}: {e}")

    vecnorm_src = artifact.get("vecnorm_path")
    if vecnorm_src and os.path.exists(vecnorm_src):
        try:
            vecnorm_dst = os.path.join(save_dir, f"{prefix}_vecnorm.pkl")
            if os.path.abspath(vecnorm_src) != os.path.abspath(vecnorm_dst):
                shutil.copyfile(vecnorm_src, vecnorm_dst)
        except Exception as e:
            logging.warning(f"[QC SAVE] vecnorm issue for {prefix}: {e}")
    else:
        logging.warning(f"[QC SAVE] vecnorm missing for {prefix}")

    try:
        with open(os.path.join(save_dir, f"{prefix}_features.json"), "w") as f:
            json.dump({"features": artifact.get("features", [])}, f)
    except Exception as e:
        logging.warning(f"[QC SAVE] features.json issue for {prefix}: {e}")

    try:
        thr = float(artifact.get("result", {}).get("Action_Threshold", 0.2))
        with open(os.path.join(save_dir, f"{prefix}_probability_config.json"), "w") as f:
            json.dump({"threshold": thr, "use_confidence": True, "inference_mode": "deterministic"}, f)
    except Exception as e:
        logging.warning(f"[QC SAVE] prob_config issue for {prefix}: {e}")

    # save gate artifacts too
    gate_model_src = artifact.get("gate_model_path")
    gate_features_src = artifact.get("gate_features_path")
    gate_config_src = artifact.get("gate_config_path")

    if gate_model_src and os.path.exists(gate_model_src):
        try:
            shutil.copyfile(gate_model_src, os.path.join(save_dir, f"{prefix}_gate_model.pkl"))
        except Exception as e:
            logging.warning(f"[QC SAVE] gate model issue for {prefix}: {e}")

    if gate_features_src and os.path.exists(gate_features_src):
        try:
            shutil.copyfile(gate_features_src, os.path.join(save_dir, f"{prefix}_gate_features.json"))
        except Exception as e:
            logging.warning(f"[QC SAVE] gate features issue for {prefix}: {e}")

    if gate_config_src and os.path.exists(gate_config_src):
        try:
            shutil.copyfile(gate_config_src, os.path.join(save_dir, f"{prefix}_gate_config.json"))
        except Exception as e:
            logging.warning(f"[QC SAVE] gate config issue for {prefix}: {e}")

    try:
        r = artifact.get("result", {})
        with open(os.path.join(save_dir, f"{prefix}_model_info.json"), "w") as f:
            json.dump(
                {
                    "model": "PPO",
                    "ticker": r.get("Ticker"),
                    "window": r.get("Window"),
                    "date_trained_utc": now_utc().strftime("%Y-%m-%dT%H:%M:%SZ"),
                    "framework": "stable-baselines3",
                    "input_features": artifact.get("features", []),
                    "final_portfolio": r.get("PPO_Portfolio"),
                    "buy_hold": r.get("BuyHold"),
                    "sharpe": r.get("Sharpe"),
                    "gate_model": r.get("Gate_Model"),
                    "gate_threshold": r.get("Gate_Threshold"),
                },
                f,
                indent=2,
            )
    except Exception as e:
        logging.warning(f"[QC SAVE] model_info issue for {prefix}: {e}")

    logging.info(f"[QC SAVE] Saved QC artifacts for {prefix} -> {save_dir}")


# -------------------------
# Parameter profiles
# -------------------------
TOP_N_WINDOWS = 3

FAST = {"lr": 8e-5, "n_steps": 3072, "batch": 512, "clip": 0.2, "ent": 0.01}
SLOW = {"lr": 3e-5, "n_steps": 3072, "batch": 512, "clip": 0.16, "ent": 0.005}

fast_names = {
    "TSLA", "NVDA", "AMD", "AVGO", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "ADBE", "CRM",
    "INTC", "QCOM", "TXN", "ORCL", "NEE", "GE", "XOM", "CVX", "LLY", "NKE", "SBUX"
}
slow_names = {
    "BRK-B", "JPM", "BAC", "JNJ", "UNH", "MRK", "PFE", "ABBV", "ABT", "AMGN", "PG", "PEP", "KO",
    "V", "MA", "WMT", "MCD", "TMO", "DHR", "ACN", "IBM", "LIN", "PM", "RTX", "UPS", "UNP", "COST", "HD", "LOW"
}


def pick_params(symbol: str) -> dict:
    return FAST if symbol in fast_names else SLOW


# -------------------------
# QC_TOP export from existing runs
# -------------------------
def export_qc_top_from_existing(ticker: str, top_n: int = 3):
    run_dirs = sorted(glob.glob(os.path.join(BASE_RESULTS_DIR, "ppo_walkforward_results_*")))[::-1]
    summary_files = []

    for rd in run_dirs:
        cand = glob.glob(os.path.join(rd, "summary*.csv"))
        if not cand:
            continue

        ok = False
        for p in cand:
            try:
                tmp = pd.read_csv(p, usecols=["Ticker"])
                if (tmp["Ticker"] == ticker).any():
                    ok = True
                    break
            except Exception:
                pass

        if ok:
            summary_files = cand
            break

    if not summary_files:
        logging.warning(f"[QC_TOP] No summaries containing {ticker} found.")
        return

    frames = []
    for p in summary_files:
        try:
            frames.append(pd.read_csv(p))
        except Exception as e:
            logging.warning(f"[QC_TOP] Could not read {p}: {e}")

    if not frames:
        logging.warning(f"[QC_TOP] No readable summaries for {ticker}.")
        return

    combo = pd.concat(frames, ignore_index=True)
    if "Ticker" not in combo.columns or "Sharpe" not in combo.columns:
        logging.warning("[QC_TOP] Missing required columns in summaries.")
        return

    combo = combo[combo["Ticker"] == ticker].copy()
    combo["Sharpe"] = pd.to_numeric(combo["Sharpe"], errors="coerce")
    combo = combo.dropna(subset=["Sharpe"])
    if combo.empty:
        logging.warning(f"[QC_TOP] No numeric Sharpe rows for {ticker}.")
        return

    use_prefix = ("Prefix" in combo.columns) and combo["Prefix"].notna().any()
    if use_prefix:
        top = combo.sort_values("Sharpe", ascending=False).head(top_n).copy()
        top["__prefix__"] = top["Prefix"].astype(str)
    else:
        def _window_start(w):
            try:
                s = str(w)
                return int(s.split("-")[0]) if "-" in s else np.nan
            except Exception:
                return np.nan

        combo["WindowStart"] = combo["Window"].apply(_window_start)
        combo = combo.sort_values(["WindowStart"]).reset_index(drop=True)
        combo["WindowIdx"] = combo.groupby("Ticker").cumcount() + 1
        top = combo.sort_values("Sharpe", ascending=False).head(top_n).copy()
        top["__prefix__"] = top["WindowIdx"].apply(lambda widx: f"ppo_{ticker}_window{int(widx)}")

    exported = 0
    for _, r in top.iterrows():
        prefix = str(r["__prefix__"])
        model_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_model.zip")
        vec_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_vecnorm.pkl")
        gate_model_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_model.pkl")
        gate_features_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_features.json")
        gate_config_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_config.json")

        if not (os.path.exists(model_path) and os.path.exists(vec_path)):
            logging.warning(f"[QC_TOP] Missing model/vecnorm for {prefix}; skip export.")
            continue

        artifact_for_save = {
            "model": None,
            "model_path": model_path,
            "vecnorm_path": vec_path,
            "features": [],
            "result": r.to_dict(),
            "prefix": prefix,
            "gate_model_path": gate_model_path if os.path.exists(gate_model_path) else None,
            "gate_features_path": gate_features_path if os.path.exists(gate_features_path) else None,
            "gate_config_path": gate_config_path if os.path.exists(gate_config_path) else None,
        }
        save_quantconnect_model(artifact_for_save, prefix, QC_TOP_DIR)
        exported += 1

    logging.info(f"[QC_TOP] Exported {exported}/{len(top)} for {ticker}")


def make_env_df(df_window: pd.DataFrame) -> pd.DataFrame:
    if "Close" not in df_window.columns:
        raise ValueError("make_env_df: missing Close")

    numeric_cols = [c for c in df_window.columns if pd.api.types.is_numeric_dtype(df_window[c])]
    cols = ["Close"] + [c for c in numeric_cols if c != "Close"]

    out = df_window[cols].copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna().reset_index(drop=True)

    if len(out) < 60:
        raise ValueError(f"make_env_df: too few rows after numeric coercion/dropna ({len(out)})")

    return out


# -------------------------
# Walkforward PPO + Gate
# -------------------------
WINDOW_SIZE = 3500
STEP_SIZE = 500
TIMESTEPS = 150_000

if TEST_MODE:
    TIMESTEPS = 50_000
    WINDOW_SIZE = 2000
    STEP_SIZE = 750


def walkforward_ppo(
    df_sym: pd.DataFrame,
    ticker: str,
    window_size: int,
    step_size: int,
    timesteps: int,
    ppo_overrides: dict,
):
    if len(df_sym) < window_size:
        logging.warning(f"Skipping {ticker}: only {len(df_sym)} rows (<{window_size}).")
        return []

    df_sym = df_sym.sort_values("Datetime").reset_index(drop=True)

    windows = get_walk_forward_windows(df_sym, window_size=window_size, step_size=step_size)
    if TEST_MODE:
        windows = windows[:1]

    results = []
    top_heap = []
    skipped_windows = []

    all_done = True
    for idx in range(len(windows)):
        prefix = f"ppo_{ticker}_window{idx + 1}"
        model_ok = os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_model.zip"))
        vec_ok = os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_vecnorm.pkl"))
        feat_ok = os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_features.json"))
        prob_ok = os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_probability_config.json"))
        info_ok = os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_model_info.json"))
        gate_ok = (
            os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_model.pkl")) and
            os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_features.json")) and
            os.path.exists(os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_config.json"))
        )
        if not (model_ok and vec_ok and feat_ok and prob_ok and info_ok and gate_ok):
            all_done = False
            break

    if all_done and len(windows) > 0 and (not FORCE_RETRAIN):
        logging.info(f"{ticker} fully skipped (all {len(windows)} windows already complete).")
        record_skips_global(ticker, [], total_windows=len(windows), fully_skipped=True)
        export_qc_top_from_existing(ticker, top_n=TOP_N_WINDOWS)
        return []

    for w_idx, (start, end) in enumerate(windows):
        window_start_time = time.time()
        gc.collect()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

        prefix = f"ppo_{ticker}_window{w_idx + 1}"
        model_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_model.zip")
        vecnorm_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_vecnorm.pkl")

        feat_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_features.json")
        prob_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_probability_config.json")
        info_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_model_info.json")

        gate_model_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_model.pkl")
        gate_features_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_features.json")
        gate_config_path = os.path.join(FINAL_MODEL_DIR, f"{prefix}_gate_config.json")

        core_ok = os.path.exists(model_path) and os.path.exists(vecnorm_path)
        master_ok = os.path.exists(feat_path) and os.path.exists(prob_path) and os.path.exists(info_path)
        gate_ok = os.path.exists(gate_model_path) and os.path.exists(gate_features_path) and os.path.exists(gate_config_path)

        if core_ok and master_ok and gate_ok and (not FORCE_RETRAIN):
            logging.info(f"Skipping {ticker} window {w_idx + 1}: already trained + MASTER + GATE complete.")
            skipped_windows.append(f"{ticker}_window{w_idx + 1}")
            manifest_update(prefix, {"status": "skipped_complete", "ticker": ticker, "window_idx": int(w_idx + 1)})
            continue

        if FORCE_RETRAIN and (core_ok or master_ok or gate_ok):
            logging.info(f"[FORCE_RETRAIN] Rebuilding {prefix} despite existing artifacts.")

        if core_ok and (not master_ok or not gate_ok) and (not FORCE_RETRAIN):
            logging.warning(f"[MASTER/GATE] {prefix} core exists but deploy artifacts incomplete; will rebuild after eval.")

        df_window = df_sym.iloc[start:end].reset_index(drop=True)

        # Window-local features
        df_window["Ret_1"] = df_window["Close"].pct_change().fillna(0.0)
        df_window["Vol_Roll"] = df_window["Ret_1"].rolling(VOL_LOOKBACK).std().fillna(0.0)
        df_window["Vol_EWM"] = df_window["Ret_1"].ewm(span=VOL_EWM_SPAN, adjust=False).std().fillna(0.0)

        vol_ref = float(df_window["Vol_EWM"].median()) if len(df_window) else 0.0
        vol_gate = float(df_window["Vol_EWM"].quantile(VOL_GATE_Q)) if len(df_window) else 0.0
        df_window["Vol_Ref"] = vol_ref
        df_window["Vol_Gate"] = vol_gate

        df_window["Ret20"] = df_window["Close"].pct_change(20).fillna(0.0)
        df_window["Ret1"] = df_window["Close"].pct_change(1).fillna(0.0)
        flip = (np.sign(df_window["Ret1"]) != np.sign(df_window["Ret1"].shift(1))).astype(float)
        df_window["FlipRate20"] = flip.rolling(20).mean().fillna(0.0)

        trend_thresh = (
            df_window["Ret20"]
            .abs()
            .rolling(200)
            .quantile(0.35)
            .fillna(df_window["Ret20"].abs().quantile(0.35))
        )

        flip_thresh = (
            df_window["FlipRate20"]
            .rolling(200)
            .quantile(0.75)
            .fillna(df_window["FlipRate20"].quantile(0.75))
        )

        trend_low = df_window["Ret20"].abs() < trend_thresh
        flip_high = df_window["FlipRate20"] > flip_thresh
        df_window["Chop"] = (trend_low & flip_high).astype(int)

        # Gate labels
        df_window = build_trade_gate_label(
            df_window,
            horizon=GATE_LABEL_HORIZON,
            ret_thr=GATE_LABEL_RETURN_THR,
        )

        if len(df_window) <= 52 or (len(df_window) % 2 != 0):
            df_window = df_window.iloc[:-1].reset_index(drop=True)

        # Gate setup
        gate_model = None
        gate_train_stats = {
            "auc": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "train_rows": 0,
            "eval_rows": 0,
            "train_pos_rate": np.nan,
            "eval_pos_rate": np.nan,
        }
        gate_features = choose_gate_features(df_window)
        gate_config = {
            "gate_enabled": bool(ENABLE_TRADE_GATE),
            "gate_model_type": GATE_MODEL_KIND,
            "threshold": float(GATE_THRESHOLD),
            "label_horizon": int(GATE_LABEL_HORIZON),
            "label_return_threshold": float(GATE_LABEL_RETURN_THR),
            "uses_scaler": False,
        }

        if ENABLE_TRADE_GATE:
            try:
                gate_fit = train_trade_gate(
                    df_gate=df_window,
                    gate_features=gate_features,
                    threshold=GATE_THRESHOLD,
                )
                gate_model = gate_fit["model"]
                gate_train_stats = {
                    "auc": gate_fit["auc"],
                    "precision": gate_fit["precision"],
                    "recall": gate_fit["recall"],
                    "train_rows": gate_fit["train_rows"],
                    "eval_rows": gate_fit["eval_rows"],
                    "train_pos_rate": gate_fit["train_pos_rate"],
                    "eval_pos_rate": gate_fit["eval_pos_rate"],
                }
                gate_config.update({
                    "train_rows": int(gate_fit["train_rows"]),
                    "eval_rows": int(gate_fit["eval_rows"]),
                    "auc": gate_fit["auc"],
                    "precision": gate_fit["precision"],
                    "recall": gate_fit["recall"],
                    "train_pos_rate": gate_fit["train_pos_rate"],
                    "eval_pos_rate": gate_fit["eval_pos_rate"],
                })
                gate_model_path, gate_features_path, gate_config_path = save_gate_model(
                    prefix=prefix,
                    gate_model=gate_model,
                    gate_features=gate_features,
                    gate_config=gate_config,
                )
            except Exception as e:
                logging.warning(f"[GATE] {prefix} gate training failed; gating disabled for this window. Err: {e}")
                gate_model = None

        df_env = make_env_df(df_window)
        frame_bound = (50, len(df_env) - 3)
        if frame_bound[1] <= frame_bound[0]:
            raise ValueError(f"{prefix}: bad frame_bound={frame_bound} with len(df_env)={len(df_env)}")

        if "Datetime" in df_window.columns:
            df_align = df_window[["Datetime", "Close"]].copy()
            df_align["Datetime"] = pd.to_datetime(df_align["Datetime"], errors="coerce")
        else:
            df_align = pd.DataFrame({"Datetime": [pd.NaT] * len(df_window), "Close": df_window["Close"].values})

        df_align["Close"] = pd.to_numeric(df_align["Close"], errors="coerce")
        df_align = df_align.dropna(subset=["Close"]).reset_index(drop=True)

        # Gate live feature frame aligned to evaluation rows
        df_gate_live = df_window[gate_features].copy()
        for c in gate_features:
            df_gate_live[c] = pd.to_numeric(df_gate_live[c], errors="coerce")
        df_gate_live = df_gate_live.reset_index(drop=True)

        n = min(len(df_env), len(df_align), len(df_gate_live))
        df_env = df_env.iloc[:n].reset_index(drop=True)
        df_align = df_align.iloc[:n].reset_index(drop=True)
        df_gate_live = df_gate_live.iloc[:n].reset_index(drop=True)

        if len(df_env) < 60:
            raise ValueError(f"{prefix}: df_env too short after alignment ({len(df_env)})")

        env = DummyVecEnv([lambda: ContinuousPositionEnv(df=df_env, frame_bound=frame_bound, **ENV_KWARGS)])
        env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)

        model = PPO(
            "MlpPolicy",
            env,
            verbose=0,
            device=DEVICE_STR,
            learning_rate=float(ppo_overrides.get("lr", 1e-4)),
            n_steps=int(ppo_overrides.get("n_steps", 256)),
            batch_size=int(ppo_overrides.get("batch", 64)),
            n_epochs=5,
            gamma=0.99,
            gae_lambda=0.95,
            clip_range=float(ppo_overrides.get("clip", 0.2)),
            ent_coef=float(ppo_overrides.get("ent", 0.005)),
            policy_kwargs=dict(net_arch=[64, 64]),
        )

        logging.info(
            f"Training {ticker} window {w_idx + 1}/{len(windows)} | "
            f"timesteps={timesteps} | lr={ppo_overrides.get('lr')}"
        )
        model.learn(total_timesteps=int(timesteps))

        env.training = False
        env.norm_reward = False

        obs = env.reset()

        nav_track = [1.0]
        bh_track = [1.0]
        step_log = []

        trades = []
        open_trade = None
        prev_pos = 0.0
        eps = 1e-9
        START_EQUITY = 100_000.0

        executed_trade_count = 0
        signal_trade_count_fixed = 0
        DIAG_THR = 0.2
        last_eval_i = 0

        gate_pass_count = 0
        gate_block_count = 0
        gate_probs_eval = []
        gate_true_eval = []

        def _safe_float(x, default=0.0):
            try:
                if x is None:
                    return float(default)
                return float(x)
            except Exception:
                return float(default)

        def close_trade(reason: str, exit_idx: int, exit_dt, exit_px: float, exit_nav: float):
            nonlocal open_trade, trades
            if open_trade is None:
                return

            entry_px = float(open_trade["EntryPrice"])
            entry_pos = float(open_trade["EntryPos"])
            side = open_trade["Side"]
            dir_sign = 1.0 if side == "LONG" else -1.0

            gross_ret = dir_sign * abs(entry_pos) * ((exit_px / max(entry_px, 1e-9)) - 1.0)
            costs = float(open_trade["Costs"])
            net_ret = gross_ret - costs
            pnl_usd = (float(exit_nav) - float(open_trade["EntryNAV"])) * START_EQUITY

            open_trade.update({
                "ExitIndex": int(exit_idx),
                "ExitDatetime": exit_dt,
                "ExitPrice": float(exit_px),
                "ExitNAV": float(exit_nav),
                "BarsHeld": int(exit_idx - int(open_trade["EntryIndex"])),
                "GrossReturnPct": float(gross_ret * 100.0),
                "CostsPct": float(costs * 100.0),
                "NetReturnPct": float(net_ret * 100.0),
                "PnL_Usd": float(pnl_usd),
                "ExitReason": str(reason),
                "MFE_Pct": float(open_trade.get("MFE", 0.0) * 100.0),
                "MAE_Pct": float(open_trade.get("MAE", 0.0) * 100.0),
            })
            trades.append(open_trade)
            open_trade = None

        if len(df_align) < 2:
            raise ValueError(f"{prefix}: df_align too short after alignment ({len(df_align)})")

        for i in range(len(df_align) - 1):
            raw_action, _ = model.predict(obs, deterministic=True)
            mu, sigma = get_mu_sigma(model, obs)

            action = raw_action
            gate_prob = np.nan
            gate_passed = True

            if ENABLE_TRADE_GATE and gate_model is not None and (i + 1) < len(df_gate_live):
                feature_row = df_gate_live.iloc[[i + 1]].copy()

                if not feature_row.isna().any(axis=None):
                    gate_prob = gate_predict_row(gate_model, feature_row)
                    gate_passed = bool(gate_prob > GATE_THRESHOLD)
                else:
                    gate_prob = np.nan
                    gate_passed = False

                gate_true = (
                    int(df_window["Trade_Label"].iloc[i + 1])
                    if "Trade_Label" in df_window.columns and (i + 1) < len(df_window)
                    else np.nan
                )
                if pd.notna(gate_prob) and pd.notna(gate_true):
                    gate_probs_eval.append(float(gate_prob))
                    gate_true_eval.append(int(gate_true))

                if gate_passed:
                    gate_pass_count += 1
                else:
                    gate_block_count += 1
                    action = np.array([0.0], dtype=np.float32)

            obs, rewards, dones, infos = env.step(action)

            rew0 = float(rewards[0]) if isinstance(rewards, np.ndarray) else float(rewards)
            done0 = bool(dones[0]) if isinstance(dones, np.ndarray) else bool(dones)
            info0 = infos[0] if isinstance(infos, (list, tuple)) and len(infos) else (infos or {})
            info0 = info0 or {}

            nav_now = _safe_float(info0.get("nav", nav_track[-1]), nav_track[-1])
            base_ret = _safe_float(info0.get("base_ret", 0.0), 0.0)
            pos_now = _safe_float(info0.get("pos", 0.0), 0.0)
            trade_cost_step = _safe_float(info0.get("trade_cost", 0.0), 0.0)
            turnover_step = _safe_float(
                info0.get("turnover_step", abs(_safe_float(info0.get("delta_pos", 0.0), 0.0))),
                0.0,
            )
            delta_pos = _safe_float(info0.get("delta_pos", 0.0), 0.0)

            nav_track.append(nav_now)
            bh_track.append(bh_track[-1] * (1.0 + base_ret))

            a = float(np.array(action).squeeze())
            raw_a = float(np.array(raw_action).squeeze())

            dt_val = df_align["Datetime"].iloc[i + 1] if "Datetime" in df_align.columns else pd.NaT
            px = float(df_align["Close"].iloc[i + 1])

            if open_trade is not None:
                open_trade["Costs"] += trade_cost_step

                entry_px = float(open_trade["EntryPrice"])
                entry_pos = float(open_trade["EntryPos"])
                side = str(open_trade["Side"])
                dir_sign = 1.0 if side == "LONG" else -1.0

                gross_ret_now = dir_sign * abs(entry_pos) * ((px / max(entry_px, 1e-9)) - 1.0)
                open_trade["MFE"] = max(float(open_trade.get("MFE", gross_ret_now)), gross_ret_now)
                open_trade["MAE"] = min(float(open_trade.get("MAE", gross_ret_now)), gross_ret_now)

            was_flat = abs(prev_pos) < eps
            now_flat = abs(pos_now) < eps
            sign_flip = (abs(prev_pos) >= eps) and (abs(pos_now) >= eps) and (np.sign(prev_pos) != np.sign(pos_now))

            if (not was_flat) and now_flat:
                close_trade(
                    reason=(
                        "TP/SL" if bool(info0.get("forced_close", False))
                        else "SLO" if bool(info0.get("slo_triggered", False))
                        else "FLAT"
                    ),
                    exit_idx=i + 1,
                    exit_dt=dt_val,
                    exit_px=px,
                    exit_nav=nav_now,
                )

            elif sign_flip:
                close_trade(
                    reason="SIGN_FLIP",
                    exit_idx=i + 1,
                    exit_dt=dt_val,
                    exit_px=px,
                    exit_nav=nav_now,
                )

                side = "LONG" if pos_now > 0 else "SHORT"
                open_trade = {
                    "Ticker": ticker,
                    "Window": f"{start}-{end}",
                    "WindowIdx": int(w_idx + 1),
                    "Prefix": prefix,
                    "Side": side,
                    "EntryIndex": int(i + 1),
                    "EntryDatetime": dt_val,
                    "EntryPrice": float(px),
                    "EntryPos": float(pos_now),
                    "EntryNAV": float(nav_now),
                    "Costs": 0.0,
                    "MFE": 0.0,
                    "MAE": 0.0,
                }

            elif was_flat and (not now_flat):
                side = "LONG" if pos_now > 0 else "SHORT"
                open_trade = {
                    "Ticker": ticker,
                    "Window": f"{start}-{end}",
                    "WindowIdx": int(w_idx + 1),
                    "Prefix": prefix,
                    "Side": side,
                    "EntryIndex": int(i + 1),
                    "EntryDatetime": dt_val,
                    "EntryPrice": float(px),
                    "EntryPos": float(pos_now),
                    "EntryNAV": float(nav_now),
                    "Costs": 0.0,
                    "MFE": 0.0,
                    "MAE": 0.0,
                }

            prev_pos = pos_now

            if (raw_a > DIAG_THR) or (raw_a < -DIAG_THR):
                signal_trade_count_fixed += 1
            if bool(info0.get("executed_trade", False)):
                executed_trade_count += 1

            if i + 2 < len(df_align):
                p0 = float(df_align["Close"].iloc[i + 1])
                p1 = float(df_align["Close"].iloc[i + 2])
                next_ret = 0.0 if p0 <= 0 else (p1 - p0) / p0
            else:
                next_ret = 0.0

            step_log.append({
                "Index": i + 1,
                "Datetime": dt_val,
                "Close": px,
                "Raw_Action": raw_a,
                "Action": a,
                "mu": mu,
                "sigma": sigma,
                "nav": nav_now,
                "ret_t": float(info0.get("ret_t", 0.0)),
                "next_ret": float(next_ret),
                "reward": float(rew0),
                "pos": float(pos_now),
                "trade_cost": float(trade_cost_step),
                "turnover_step": float(turnover_step),
                "delta_pos": float(delta_pos),
                "base_ret": float(base_ret),
                "rel_alpha": float(info0.get("rel_alpha", 0.0)),
                "mom": float(info0.get("mom", 0.0)),
                "vol_t": float(info0.get("vol_t", 0.0)),
                "vol_ref": float(info0.get("vol_ref", 0.0)),
                "vol_gate": float(info0.get("vol_gate", 0.0)),
                "vol_scale": float(info0.get("vol_scale", 1.0)),
                "vol_gated": bool(info0.get("vol_gated", False)),
                "tp_pct": float(info0.get("tp_pct", 0.0)),
                "sl_pct": float(info0.get("sl_pct", 0.0)),
                "forced_close": bool(info0.get("forced_close", False)),
                "slo_triggered": bool(info0.get("slo_triggered", False)),
                "slo_locked": bool(info0.get("slo_locked", False)),
                "slo_lock_until": int(info0.get("slo_lock_until", -1)),
                "drawdown": float(info0.get("drawdown", 0.0)),
                "chop_locked": bool(info0.get("chop_locked", False)),
                "Gate_Prob": gate_prob,
                "Gate_Passed": gate_passed,
                "Gate_Blocked": (not gate_passed) if ENABLE_TRADE_GATE else False,
            })

            last_eval_i = i + 1
            if done0:
                break

        if open_trade is not None:
            last_idx = int(min(last_eval_i, len(df_align) - 1))
            last_dt = df_align["Datetime"].iloc[last_idx] if "Datetime" in df_align.columns else pd.NaT
            last_px = float(df_align["Close"].iloc[last_idx])
            last_nav = float(nav_track[-1])
            close_trade(reason="EOD", exit_idx=last_idx, exit_dt=last_dt, exit_px=last_px, exit_nav=last_nav)

        # -------------------------
        # Metrics
        # -------------------------
        step_df = pd.DataFrame(step_log)

        avg_abs_exposure = float(step_df["pos"].abs().mean()) if len(step_df) else 0.0
        turnover = float(step_df["turnover_step"].sum()) if ("turnover_step" in step_df.columns and len(step_df)) else 0.0
        avg_bars_held = float(
            pd.to_numeric(pd.DataFrame(trades).get("BarsHeld", pd.Series(dtype=float)), errors="coerce").dropna().mean()
        ) if len(trades) else 0.0

        vol_gated_count = int(step_df["vol_gated"].sum()) if "vol_gated" in step_df.columns else 0
        forced_close_count = int(step_df["forced_close"].sum()) if "forced_close" in step_df.columns else 0
        slo_trigger_count = int(step_df["slo_triggered"].sum()) if "slo_triggered" in step_df.columns else 0
        chop_locked_count = int(step_df["chop_locked"].sum()) if "chop_locked" in step_df.columns else 0

        chop_rate = float(step_df["chop_locked"].mean()) if "chop_locked" in step_df.columns and len(step_df) else 0.0

        logging.info(
            f"[REGIME CHECK] {prefix} | chop_rate={chop_rate:.4f} | "
            f"vol_gated_count={vol_gated_count} | forced_close_count={forced_close_count} | "
            f"slo_trigger_count={slo_trigger_count}"
        )

        abs_actions = np.array([abs(float(r["Action"])) for r in step_log], dtype=float)
        if len(abs_actions) > 0:
            thr_window = float(np.quantile(abs_actions, 0.60))
            thr_window = float(np.clip(thr_window, 0.08, 0.25))

            logging.info(
                f"[THRESHOLD CHECK] {prefix} | thr_window={thr_window:.4f} | "
                f"min_abs_action={abs_actions.min():.4f} | "
                f"median_abs_action={np.median(abs_actions):.4f} | "
                f"max_abs_action={abs_actions.max():.4f}"
            )
        else:
            thr_window = 0.2
            logging.info(f"[THRESHOLD CHECK] {prefix} | no actions found, using default thr_window=0.2000")

        # Gate metrics from eval loop
        gate_auc_eval = np.nan
        gate_precision_eval = np.nan
        gate_recall_eval = np.nan
        gate_pass_rate = np.nan

        if ENABLE_TRADE_GATE and (gate_pass_count + gate_block_count) > 0:
            gate_pass_rate = float(gate_pass_count / max(gate_pass_count + gate_block_count, 1))

        if ENABLE_TRADE_GATE and len(gate_probs_eval) > 0:
            gp = np.array(gate_probs_eval, dtype=float)
            gy = np.array(gate_true_eval, dtype=int)
            gpred = (gp > GATE_THRESHOLD).astype(int)

            if len(np.unique(gy)) > 1:
                try:
                    gate_auc_eval = float(roc_auc_score(gy, gp))
                except Exception:
                    pass
            try:
                gate_precision_eval = float(precision_score(gy, gpred, zero_division=0))
            except Exception:
                pass
            try:
                gate_recall_eval = float(recall_score(gy, gpred, zero_division=0))
            except Exception:
                pass

        correct = 0
        total = 0
        tp_buy = fp_buy = tp_sell = fp_sell = 0

        for r in step_log:
            a = float(r["Action"])
            ret_t = float(r.get("next_ret", 0.0))

            if a > thr_window:
                total += 1
                if ret_t > 0:
                    tp_buy += 1
                    correct += 1
                else:
                    fp_buy += 1
            elif a < -thr_window:
                total += 1
                if ret_t < 0:
                    tp_sell += 1
                    correct += 1
                else:
                    fp_sell += 1

        precision_long = tp_buy / (tp_buy + fp_buy + 1e-9)
        precision_short = tp_sell / (tp_sell + fp_sell + 1e-9)
        precision_trades = (tp_buy + tp_sell) / ((tp_buy + tp_sell) + (fp_buy + fp_sell) + 1e-9)
        step_accuracy = round(correct / total, 4) if total > 0 else 0.0

        final_value = float(nav_track[-1]) * 100_000.0
        hold_value = float(bh_track[-1]) * 100_000.0

        returns = pd.Series(nav_track).pct_change().fillna(0.0)
        sharpe = float((returns.mean() / (returns.std() + 1e-9)) * _annualization_factor())
        drawdown = float(
            (
                (pd.Series(nav_track).cummax() - pd.Series(nav_track))
                / pd.Series(nav_track).cummax()
            ).max() * 100.0
        )

        logging.info(
            f"[GATE CHECK] {prefix} | pass_count={gate_pass_count} | block_count={gate_block_count} | "
            f"gate_auc_eval={gate_auc_eval} | gate_precision_eval={gate_precision_eval} | gate_recall_eval={gate_recall_eval}"
        )

        logging.info(
            f"[EVAL] {prefix} executed_trade_count={executed_trade_count} "
            f"avg_abs_exposure={avg_abs_exposure:.4f} final_nav={nav_track[-1]:.6f}"
        )

        # -------------------------
        # Save artifacts
        # -------------------------
        trades_path = os.path.join(RUN_RESULTS_DIR, f"{prefix}_trades.csv")
        pd.DataFrame(trades).to_csv(trades_path, index=False)

        model.save(model_path)
        try:
            env.save(vecnorm_path)
        except Exception as e:
            logging.warning(f"VecNormalize save failed for {prefix}: {e}")
            try:
                if os.path.exists(model_path):
                    os.remove(model_path)
            except Exception:
                pass
            raise

        exclude_feats = {"Datetime", "Symbol", "Target", "Return", "Trade_Label", "Gate_Future_Return"}
        feature_cols = [
            c for c in df_window.columns
            if c not in exclude_feats and pd.api.types.is_numeric_dtype(df_window[c])
        ]

        model_info = {
            "model": "PPO",
            "ticker": ticker,
            "window": f"{start}-{end}",
            "window_idx": int(w_idx + 1),
            "prefix": prefix,
            "date_trained_utc": now_utc().strftime("%Y-%m-%dT%H:%M:%SZ"),
            "framework": "stable-baselines3",
            "device": DEVICE_STR,
            "seed": SEED,
            "timesteps": int(timesteps),
            "feature_pipeline": {
                "fallback_active": FEATURES_FALLBACK_ACTIVE,
                "module_import_ok": not FEATURES_FALLBACK_ACTIVE,
            },
            "env_kwargs": dict(ENV_KWARGS),
            "ppo_overrides": dict(ppo_overrides),
            "gate_info": {
                "enabled": bool(ENABLE_TRADE_GATE and gate_model is not None),
                "model_type": GATE_MODEL_KIND,
                "threshold": float(GATE_THRESHOLD),
                "label_horizon": int(GATE_LABEL_HORIZON),
                "label_return_threshold": float(GATE_LABEL_RETURN_THR),
                "features_used": list(gate_features),
                "train_rows": int(gate_train_stats["train_rows"]),
                "eval_rows": int(gate_train_stats["eval_rows"]),
                "train_auc": gate_train_stats["auc"],
                "train_precision": gate_train_stats["precision"],
                "train_recall": gate_train_stats["recall"],
                "eval_auc": gate_auc_eval,
                "eval_precision": gate_precision_eval,
                "eval_recall": gate_recall_eval,
                "pass_count": int(gate_pass_count),
                "block_count": int(gate_block_count),
            },
            "metrics": {
                "final_portfolio": round(final_value, 2),
                "buy_hold": round(hold_value, 2),
                "sharpe": round(sharpe, 6),
                "drawdown_pct": round(drawdown, 6),
                "trade_count": int(executed_trade_count),
                "action_threshold": float(thr_window),
                "accuracy": float(step_accuracy),
                "precision_long": float(precision_long),
                "precision_short": float(precision_short),
                "precision_trades": float(precision_trades),
                "avg_abs_exposure": round(avg_abs_exposure, 6),
                "turnover": round(turnover, 6),
                "avg_bars_held": round(avg_bars_held, 3),
                "vol_gated_count": int(vol_gated_count),
                "forced_close_count": int(forced_close_count),
                "slo_trigger_count": int(slo_trigger_count),
                "chop_locked_count": int(chop_locked_count),
                "gate_pass_count": int(gate_pass_count),
                "gate_block_count": int(gate_block_count),
                "gate_pass_rate": gate_pass_rate,
                "gate_auc_eval": gate_auc_eval,
                "gate_precision_eval": gate_precision_eval,
                "gate_recall_eval": gate_recall_eval,
            }
        }

        write_master_bundle(
            prefix=prefix,
            features=feature_cols,
            threshold=float(thr_window),
            model_info=model_info,
            gate_features=gate_features if gate_model is not None else None,
            gate_threshold=float(GATE_THRESHOLD) if gate_model is not None else None,
        )

        pred_path = os.path.join(RUN_RESULTS_DIR, f"{prefix}_predictions.csv")
        pd.DataFrame(step_log).to_csv(pred_path, index=False)

        compat_rows = []
        for r in step_log:
            a = float(r["Action"])
            if a > thr_window:
                signal = "BUY"
            elif a < -thr_window:
                signal = "SELL"
            else:
                signal = "HOLD"

            compat_rows.append({
                "Index": r["Index"],
                "Datetime": r["Datetime"],
                "Close": r["Close"],
                "Raw_Action": r["Raw_Action"],
                "Action": a,
                "Signal": signal,
                "PortfolioValue": r["nav"],
                "Reward": r.get("reward", np.nan),
                "Gate_Prob": r.get("Gate_Prob", np.nan),
                "Gate_Passed": r.get("Gate_Passed", np.nan),
                "Gate_Blocked": r.get("Gate_Blocked", np.nan),
            })

        compat_path = os.path.join(RUN_RESULTS_DIR, f"{prefix}_predictions_compat.csv")
        pd.DataFrame(compat_rows).to_csv(compat_path, index=False)

        manifest_record_paths(
            prefix,
            {
                "model_zip": model_path,
                "vecnorm_pkl": vecnorm_path,
                "predictions_csv": pred_path,
                "predictions_compat_csv": compat_path,
                "trades_csv": trades_path,
                "gate_model_pkl": gate_model_path if os.path.exists(gate_model_path) else None,
                "gate_features_json": gate_features_path if os.path.exists(gate_features_path) else None,
                "gate_config_json": gate_config_path if os.path.exists(gate_config_path) else None,
            },
            do_hashes=ENABLE_HASHES,
        )

        manifest_update(
            prefix,
            {
                "status": "trained",
                "ticker": ticker,
                "window": f"{start}-{end}",
                "window_idx": int(w_idx + 1),
                "sharpe": float(sharpe),
                "gate_enabled": bool(ENABLE_TRADE_GATE and gate_model is not None),
            }
        )

        result_row = {
            "Ticker": ticker,
            "Window": f"{start}-{end}",
            "WindowIdx": int(w_idx + 1),
            "Prefix": prefix,
            "PPO_Portfolio": round(final_value, 2),
            "BuyHold": round(hold_value, 2),
            "Sharpe": round(sharpe, 3),
            "Drawdown_%": round(drawdown, 2),
            "Winner": "PPO" if final_value > hold_value else "Buy & Hold",
            "Action_Threshold": round(thr_window, 4),
            "Accuracy": step_accuracy,
            "Trade_Count": int(executed_trade_count),
            "Signal_Trade_Count": int(signal_trade_count_fixed),
            "Vol_Gated_Count": int(vol_gated_count),
            "Forced_Close_Count": int(forced_close_count),
            "SLO_Trigger_Count": int(slo_trigger_count),
            "Chop_Locked_Count": int(chop_locked_count),
            "Precision_Long": round(precision_long, 4),
            "Precision_Short": round(precision_short, 4),
            "Precision_Trades": round(precision_trades, 4),

            # Phase 2 gate output
            "Gate_Model": GATE_MODEL_KIND if gate_model is not None else "disabled",
            "Gate_Threshold": float(GATE_THRESHOLD) if gate_model is not None else np.nan,
            "Gate_Pass_Count": int(gate_pass_count),
            "Gate_Block_Count": int(gate_block_count),
            "Gate_Pass_Rate": gate_pass_rate,
            "Gate_AUC_Train": gate_train_stats["auc"],
            "Gate_Precision_Train": gate_train_stats["precision"],
            "Gate_Recall_Train": gate_train_stats["recall"],
            "Gate_AUC_Eval": gate_auc_eval,
            "Gate_Precision_Eval": gate_precision_eval,
            "Gate_Recall_Eval": gate_recall_eval,
        }

        results.append(result_row)

        meta = {
            "result": result_row,
            "features": feature_cols,
            "prefix": prefix,
            "model_path": model_path,
            "vecnorm_path": vecnorm_path,
            "gate_model_path": gate_model_path if os.path.exists(gate_model_path) else None,
            "gate_features_path": gate_features_path if os.path.exists(gate_features_path) else None,
            "gate_config_path": gate_config_path if os.path.exists(gate_config_path) else None,
        }
        item = (float(result_row["Sharpe"]), prefix, meta)
        if len(top_heap) < TOP_N_WINDOWS:
            heapq.heappush(top_heap, item)
        else:
            if item[0] > top_heap[0][0]:
                heapq.heapreplace(top_heap, item)

        logging.info(
            f"{ticker} | window {w_idx + 1} done | "
            f"sharpe={result_row['Sharpe']} | runtime={round(time.time() - window_start_time, 2)}s"
        )

        top_list = sorted(top_heap, key=lambda t: t[0], reverse=True)
        for _, _, m in top_list:
            artifact_for_save = {
                "model": None,
                "model_path": m["model_path"],
                "vecnorm_path": m["vecnorm_path"],
                "features": m["features"],
                "result": m["result"],
                "prefix": m["prefix"],
                "gate_model_path": m.get("gate_model_path"),
                "gate_features_path": m.get("gate_features_path"),
                "gate_config_path": m.get("gate_config_path"),
            }
            if GATE_TOP_EXPORT_TO_QC:
                save_quantconnect_model(artifact_for_save, m["prefix"], QC_TOP_DIR)

        try:
            env.close()
        except Exception:
            pass

        del env, model
        gc.collect()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

    if skipped_windows:
        record_skips_global(ticker, skipped_windows, total_windows=len(windows), fully_skipped=False)

    return results


# -------------------------
# Parallel runner
# -------------------------
from concurrent.futures import ThreadPoolExecutor


def process_ticker(ticker: str) -> list[dict]:
    try:
        hp = pick_params(ticker)
        df_sym = df[df["Symbol"] == ticker].copy()
        return walkforward_ppo(
            df_sym=df_sym,
            ticker=ticker,
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE,
            timesteps=TIMESTEPS,
            ppo_overrides=hp,
        )
    except Exception as e:
        logging.error(f"{ticker}: training failed: {e}")
        return []


def run_parallel_tickers(tickers: list[str], out_path: str, max_workers: int):
    logging.info(f"[PARALLEL] max_workers={max_workers}")
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for res in ex.map(process_ticker, tickers):
            if res:
                results.extend(res)

    if results:
        pd.DataFrame(results).to_csv(out_path, index=False)
        logging.info(f"Saved summary -> {out_path}")
    else:
        logging.warning("No results produced; summary not written.")

    return results


# -------------------------
# Selector builder
# -------------------------
MODEL_NAME = "PPO"


def build_ppo_selector():
    summary_files = glob.glob(os.path.join(BASE_RESULTS_DIR, "ppo_walkforward_results_*", "summary*.csv"))
    frames = []

    for p in summary_files:
        try:
            tmp = pd.read_csv(p)
            tmp["RunFolder"] = os.path.dirname(p)
            frames.append(tmp)
        except Exception as e:
            logging.warning(f"Skipping summary {p}: {e}")

    if not frames:
        logging.warning("No PPO summaries found across run folders.")
        return

    combo = pd.concat(frames, ignore_index=True)

    if "Sharpe" not in combo.columns:
        logging.warning("Combined summaries missing 'Sharpe'. Selector will be empty.")
        return

    combo["Sharpe"] = pd.to_numeric(combo["Sharpe"], errors="coerce")
    combo = combo.dropna(subset=["Sharpe"])
    if combo.empty:
        logging.warning("No numeric Sharpe rows after coercion.")
        return

    if "BuyHold" not in combo.columns:
        combo["BuyHold"] = np.nan
    if "PPO_Portfolio" not in combo.columns:
        combo["PPO_Portfolio"] = np.nan
    if "Drawdown_%" not in combo.columns:
        combo["Drawdown_%"] = np.nan

    def _parse_window_start(w):
        if pd.isna(w):
            return None
        s = str(w)
        if "-" in s:
            try:
                return int(s.split("-")[0])
            except Exception:
                return None
        try:
            return int(float(s))
        except Exception:
            return None

    combo["WindowStart"] = combo["Window"].apply(_parse_window_start)

    def _parse_run_ts(run_folder: str):
        try:
            base = os.path.basename(str(run_folder))
            parts = base.replace("ppo_walkforward_results_", "").split("_")
            if len(parts) >= 2:
                dt_str = parts[0] + parts[1]
                return pd.to_datetime(dt_str, format="%Y%m%d%H%M%S", errors="coerce")
            if len(parts) == 1:
                return pd.to_datetime(parts[0], errors="coerce")
        except Exception:
            pass
        return pd.NaT

    combo["RunTS"] = combo["RunFolder"].apply(_parse_run_ts)
    combo = combo.sort_values(["Ticker", "WindowStart", "RunTS"]).reset_index(drop=True)

    if "WindowIdx" in combo.columns:
        combo["WindowIdx"] = pd.to_numeric(combo["WindowIdx"], errors="coerce")
    else:
        combo["WindowIdx"] = combo.groupby("Ticker").cumcount() + 1

    combo = combo.drop_duplicates(subset=["Ticker", "WindowStart"], keep="last")

    best_by_symbol = (
        combo.sort_values("Sharpe", ascending=False)
        .groupby("Ticker")
        .first()
        .reset_index()
    )

    best_by_symbol = best_by_symbol.rename(columns={"Drawdown_%": "Drawdown", "PPO_Portfolio": "Final_Portfolio"})
    best_by_symbol["Model"] = MODEL_NAME

    def _safe_rel(r):
        bh = r.get("BuyHold", np.nan)
        fp = r.get("Final_Portfolio", np.nan)
        if pd.notna(bh) and float(bh) != 0.0:
            return float(fp) / float(bh)
        return np.nan

    best_by_symbol["Rel_vs_BH"] = best_by_symbol.apply(_safe_rel, axis=1)

    best_by_symbol.to_csv(SELECTOR_FULL_PATH, index=False)
    logging.info(f"Selector FULL saved -> {SELECTOR_FULL_PATH}")

    # selector already ranks on gated performance because PPO_Portfolio is now gated portfolio
    df_sel = best_by_symbol.copy()
    gates = (
        (df_sel["Sharpe"].fillna(-999) > 0.0)
        & (df_sel["Drawdown"].fillna(999) < 50.0)
        & (df_sel["Final_Portfolio"].fillna(0) > 80_000)
        & (df_sel["Rel_vs_BH"].fillna(0) >= 0.95)
    )
    df_sel = df_sel[gates].copy()

    df_sel["prefix"] = (
        "ppo_" + df_sel["Ticker"].astype(str) + "_window"
        + df_sel["WindowIdx"].fillna(1).astype(int).astype(str)
    )
    df_sel["artifact_path"] = df_sel["prefix"].apply(lambda p: os.path.join(FINAL_MODEL_DIR, f"{p}_model.zip"))
    df_sel["vecnorm_path"] = df_sel["prefix"].apply(lambda p: os.path.join(FINAL_MODEL_DIR, f"{p}_vecnorm.pkl"))
    df_sel["gate_model_path"] = df_sel["prefix"].apply(lambda p: os.path.join(FINAL_MODEL_DIR, f"{p}_gate_model.pkl"))
    df_sel["gate_features_path"] = df_sel["prefix"].apply(lambda p: os.path.join(FINAL_MODEL_DIR, f"{p}_gate_features.json"))
    df_sel["gate_config_path"] = df_sel["prefix"].apply(lambda p: os.path.join(FINAL_MODEL_DIR, f"{p}_gate_config.json"))

    selected_models = {}
    for _, row in df_sel.iterrows():
        tkr = str(row["Ticker"])
        selected_models[tkr] = {
            "model": MODEL_NAME,
            "score": float(row["Sharpe"]),
            "sharpe": float(row["Sharpe"]),
            "return": float(row["Final_Portfolio"]) if pd.notna(row["Final_Portfolio"]) else None,
            "drawdown": float(row["Drawdown"]) if pd.notna(row["Drawdown"]) else None,
            "trade_count": int(row["Trade_Count"]) if "Trade_Count" in row and pd.notna(row["Trade_Count"]) else None,
            "gate_model": row.get("Gate_Model", None),
            "gate_threshold": float(row["Gate_Threshold"]) if "Gate_Threshold" in row and pd.notna(row["Gate_Threshold"]) else None,
            "gate_pass_count": int(row["Gate_Pass_Count"]) if "Gate_Pass_Count" in row and pd.notna(row["Gate_Pass_Count"]) else None,
            "gate_block_count": int(row["Gate_Block_Count"]) if "Gate_Block_Count" in row and pd.notna(row["Gate_Block_Count"]) else None,
            "artifact": {
                "path": row["artifact_path"],
                "vecnorm": row["vecnorm_path"],
                "exists": bool(os.path.exists(row["artifact_path"])),
                "gate_model_path": row["gate_model_path"] if os.path.exists(row["gate_model_path"]) else None,
                "gate_features_path": row["gate_features_path"] if os.path.exists(row["gate_features_path"]) else None,
                "gate_config_path": row["gate_config_path"] if os.path.exists(row["gate_config_path"]) else None,
            },
        }

    with open(SELECTOR_JSON_PATH, "w") as f:
        json.dump(selected_models, f, indent=2)
    logging.info(f"Selector JSON saved -> {SELECTOR_JSON_PATH}")


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    hc = health_check_latest_run(BASE_RESULTS_DIR)
    if hc.get("ok"):
        logging.info(f"[HEALTH] Latest run folder: {hc['latest_run_dir']}")
        logging.info(f"[HEALTH] Summary files: {hc['summary_files']}")
        logging.info(f"[HEALTH] Rows total: {hc['rows_total']} | tickers: {len(hc['tickers'])}")
    else:
        logging.warning(f"[HEALTH] {hc.get('reason')}")

    logging.info(f"RUN_RESULTS_DIR   = {RUN_RESULTS_DIR}")
    logging.info(f"FINAL_MODEL_DIR   = {FINAL_MODEL_DIR}")
    logging.info(f"BASE_RESULTS_DIR  = {BASE_RESULTS_DIR}")
    logging.info(f"DATA_PATH         = {DATA_PATH}")

    min_rows = WINDOW_SIZE + 50
    counts = df["Symbol"].value_counts()
    candidate_symbols = [sym for sym, n in counts.items() if int(n) >= int(min_rows)]

    for sym, n in counts.items():
        if int(n) < int(min_rows):
            logging.warning(f"Skipping {sym}: only {n} rows (< {min_rows}).")

    if not candidate_symbols:
        logging.error("No symbols have enough rows for WINDOW_SIZE. Nothing to train.")
    else:
        logging.info(f"Candidate symbols: {candidate_symbols}")

    needed_cols = ["Close", "Datetime"]
    if ENABLE_WAVELET:
        needed_cols.append("Denoised_Close")
    if ENABLE_SENTIMENT:
        needed_cols.append("SentimentScore")

    valid_symbols = []
    for sym in candidate_symbols:
        cols = set(df.loc[df["Symbol"] == sym].columns)
        missing = [c for c in needed_cols if c not in cols]
        if missing:
            logging.warning(f"Skipping {sym}: missing required cols {missing}")
        else:
            valid_symbols.append(sym)

    if not valid_symbols:
        logging.error("No symbols passed column checks. Nothing to train.")
    else:
        logging.info(f"Final training universe: {valid_symbols}")

    if FORCE_RETRAIN and valid_symbols:
        cleanup_for_force_retrain(
            run_results_dir=RUN_RESULTS_DIR,
            final_model_dir=FINAL_MODEL_DIR,
            qc_top_dir=QC_TOP_DIR,
            tickers=valid_symbols,
            max_windows=50,
        )

        _safe_json_dump(MANIFEST, ARTIFACT_MANIFEST_PATH)
        _safe_json_dump(RUN_CONFIG, RUN_CONFIG_PATH)
        _safe_json_dump({"features_ordered": ORDERED_FEATURES}, FEATURE_MANIFEST_PATH)
        _safe_json_dump(PROVENANCE, DATA_PROVENANCE_PATH)

    all_results = []

    if TEST_MODE:
        test_stocks = ["AAPL", "NVDA", "MSFT"]
        present = [s for s in test_stocks if s in valid_symbols]
        if not present:
            logging.warning("TEST_MODE: none of ['AAPL','NVDA','MSFT'] present after filters.")
        else:
            logging.info(f"TEST_MODE running: {present}")

        for sym in present:
            logging.info(f">>> [TEST_MODE] Processing {sym}")
            res = process_ticker(sym)
            logging.info(f"{sym}: produced {len(res)} window summaries")
            if res:
                all_results.extend(res)

        summary_path = os.path.join(RUN_RESULTS_DIR, "summary_test_mode.csv")
        if all_results:
            pd.DataFrame(all_results).to_csv(summary_path, index=False)
            logging.info(f"Test-mode summary saved -> {summary_path}")
        else:
            logging.warning("Test mode finished but no results were generated.")
    else:
        summary_path = os.path.join(RUN_RESULTS_DIR, "summary.csv")
        all_results = run_parallel_tickers(valid_symbols, out_path=summary_path, max_workers=MAX_WORKERS)

    try:
        build_ppo_selector()
    except Exception as e:
        logging.error(f"build_ppo_selector failed: {e}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
2026-03-07 22:32:35,659 - WARNING - [WARN] feature_engineering import failed; fallback active. Err: No module named 'feature_engineering'
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-03-07 22:32:36,343 - INFO - [CONFIG] IN_COLAB=True | MAX_WORKERS=1 | cuda=False | TEST_MODE=True | DEVICE=cpu


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2026-03-07 22:32:36,981 - WARNING - [HEALTH] Latest run has no readable summary*.csv
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-03-07 22:32:36,982 - INFO - RUN_RESULTS_DIR   = /content/drive/MyDrive/Results_May_2025/ppo_walkforward_results_20260307_223236
2026-03-07 22:32:36,985 - INFO - FINAL_MODEL_DIR   = /content/drive/MyDrive/Results_May_2025/ppo_models_master
2026-03-07 22:32:36,989 - INFO - BASE_RESULTS_DIR  = /content/drive/MyDrive/Results_May_2025
2026-03-07 22:32:36,991 - INFO - DATA_PATH         = multi_stock_feature_engineered_dataset.csv
2026-03-07 22:32:36,998 - INFO - Candidate symbols: ['MSFT', 'AAPL', 'NVDA']
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWa

In [11]:
import os, glob, re, json
import pandas as pd
from datetime import datetime

BASE = "/content/drive/MyDrive/Results_May_2025"
MASTER_DIR = os.path.join(BASE, "ppo_models_master")
QC_DIR     = os.path.join(BASE, "ppo_models_QC_TOP")

# -------------------------
# Helpers
# -------------------------
def _count(d):
    return {
        "zip": len(glob.glob(os.path.join(d, "*.zip"))),
        "pkl": len(glob.glob(os.path.join(d, "*.pkl"))),
        "features": len(glob.glob(os.path.join(d, "*_features.json"))),
        "prob_cfg": len(glob.glob(os.path.join(d, "*_probability_config.json"))),
        "info": len(glob.glob(os.path.join(d, "*_model_info.json"))),
        "pred": len(glob.glob(os.path.join(d, "*_predictions.csv"))),
        "pred_compat": len(glob.glob(os.path.join(d, "*_predictions_compat.csv"))),
        "trades": len(glob.glob(os.path.join(d, "*_trades.csv"))),
        "summary": len(glob.glob(os.path.join(d, "summary*.csv"))),
        "manifest": int(os.path.exists(os.path.join(d, "artifact_manifest.json"))),
        "config": int(os.path.exists(os.path.join(d, "config.json"))),
        "feature_manifest": int(os.path.exists(os.path.join(d, "feature_manifest.json"))),
        "data_provenance": int(os.path.exists(os.path.join(d, "data_provenance.json"))),
    }

def _prefixes_from_models(folder):
    zips = glob.glob(os.path.join(folder, "ppo_*_model.zip"))
    prefs = sorted(set(re.sub(r"_model\.zip$", "", os.path.basename(p)) for p in zips))
    return prefs

def _latest_run_folder():
    runs = sorted(glob.glob(os.path.join(BASE, "ppo_walkforward_results_*")))
    return runs[-1] if runs else None

def _latest_summary_in_run(run_folder):
    if not run_folder:
        return None
    candidates = sorted(glob.glob(os.path.join(run_folder, "summary*.csv")))
    return candidates[-1] if candidates else None

def _latest_manifest_in_run(run_folder):
    if not run_folder:
        return None
    p = os.path.join(run_folder, "artifact_manifest.json")
    return p if os.path.exists(p) else None

def _check_deploy_bundle(prefix, folder):
    need = ["_model.zip", "_vecnorm.pkl", "_features.json", "_probability_config.json", "_model_info.json"]
    return {s: os.path.exists(os.path.join(folder, prefix + s)) for s in need}

def _check_run_outputs(prefix, run_folder):
    need = ["_predictions.csv", "_predictions_compat.csv", "_trades.csv"]
    return {s: os.path.exists(os.path.join(run_folder, prefix + s)) for s in need}

def _print_bundle_status(label, prefix, status_dict):
    ok = all(status_dict.values())
    print(f"{label} | {prefix} ->", ("OK" if ok else "MISSING"), status_dict)

def _run_state(run_folder):
    if not run_folder or not os.path.exists(run_folder):
        return "missing"
    counts = _count(run_folder)
    if counts["summary"] > 0:
        return "completed_or_partial_outputs_present"
    if counts["manifest"] or counts["config"] or counts["feature_manifest"] or counts["data_provenance"]:
        return "initialized_but_no_summary_yet"
    return "empty_or_uninitialized"

# -------------------------
# Header
# -------------------------
print("\n=== PPO ARTIFACT HEALTH CHECK (Updated) ===")
print("Time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("BASE:", BASE)
print("MASTER_DIR:", MASTER_DIR, "| exists:", os.path.exists(MASTER_DIR))
print("QC_DIR:", QC_DIR, "| exists:", os.path.exists(QC_DIR))

# -------------------------
# Latest run folder
# -------------------------
latest_run = _latest_run_folder()
latest_summary = _latest_summary_in_run(latest_run)
latest_manifest = _latest_manifest_in_run(latest_run)

print("\n[Latest Run]")
print("Latest run folder:", latest_run)
print("Latest run state:", _run_state(latest_run))
print("Latest summary file:", latest_summary)
print("Latest manifest file:", latest_manifest)

if latest_manifest:
    try:
        with open(latest_manifest, "r") as f:
            man = json.load(f)
        art = man.get("artifacts", {})
        print("Manifest artifact prefixes:", len(art))
        print("Manifest sample prefixes:", list(art.keys())[:10])
    except Exception as e:
        print("Could not read manifest:", e)

if latest_summary and os.path.exists(latest_summary):
    try:
        s = pd.read_csv(latest_summary)
        print("Summary rows:", len(s))
        if "Ticker" in s.columns:
            print("Tickers in summary:", sorted(s["Ticker"].dropna().astype(str).unique().tolist())[:50])
    except Exception as e:
        print("Could not read summary:", e)
else:
    print("No summary file in latest run yet. This can be normal right after FORCE_RETRAIN cleanup or before training finishes.")

# -------------------------
# Folder counts
# -------------------------
print("\n[Counts]")
print("MASTER counts:", _count(MASTER_DIR) if os.path.exists(MASTER_DIR) else "missing folder")
print("QC_TOP counts:", _count(QC_DIR) if os.path.exists(QC_DIR) else "missing folder")
print("RUN counts:", _count(latest_run) if (latest_run and os.path.exists(latest_run)) else "missing run folder")

# -------------------------
# Prefix lists
# -------------------------
master_prefixes = _prefixes_from_models(MASTER_DIR) if os.path.exists(MASTER_DIR) else []
qc_prefixes     = _prefixes_from_models(QC_DIR) if os.path.exists(QC_DIR) else []

print("\n[Prefixes]")
print("MASTER prefixes:", master_prefixes[:15], ("... total " + str(len(master_prefixes)) if len(master_prefixes) > 15 else ""))
print("QC_TOP prefixes:", qc_prefixes[:15], ("... total " + str(len(qc_prefixes)) if len(qc_prefixes) > 15 else ""))

# -------------------------
# QC_TOP integrity (deploy-only)
# -------------------------
print("\n[QC_TOP Deploy Bundle Integrity — should be all True]")
for pref in qc_prefixes[:10]:
    status = _check_deploy_bundle(pref, QC_DIR)
    _print_bundle_status("QC_TOP", pref, status)

# -------------------------
# MASTER integrity (deploy-only)
# -------------------------
print("\n[MASTER Deploy Bundle Integrity — should be all True]")
for pref in master_prefixes[:10]:
    status = _check_deploy_bundle(pref, MASTER_DIR)
    _print_bundle_status("MASTER", pref, status)

# -------------------------
# Run outputs integrity (run-only)
# -------------------------
if latest_run and os.path.exists(latest_run):
    print("\n[RUN Output Integrity — predictions/trades live here]")
    pred_files = glob.glob(os.path.join(latest_run, "ppo_*_predictions.csv"))
    run_prefs = sorted(set(os.path.basename(p).replace("_predictions.csv", "") for p in pred_files))
    if run_prefs:
        for pref in run_prefs[:10]:
            status = _check_run_outputs(pref, latest_run)
            _print_bundle_status("RUN", pref, status)
    else:
        print("No run-level prediction files found in latest run folder.")

# -------------------------
# One-off check function
# -------------------------
def check_prefix(prefix):
    print("\n[Check prefix]", prefix)

    if os.path.exists(os.path.join(QC_DIR, prefix + "_model.zip")):
        where = "QC_TOP"
        folder = QC_DIR
        status = _check_deploy_bundle(prefix, folder)
        print("Found in:", where, "->", folder)
        print(status)
        return

    if os.path.exists(os.path.join(MASTER_DIR, prefix + "_model.zip")):
        where = "MASTER"
        folder = MASTER_DIR
        status = _check_deploy_bundle(prefix, folder)
        print("Found in:", where, "->", folder)
        print(status)
        return

    if latest_run and os.path.exists(os.path.join(latest_run, prefix + "_predictions.csv")):
        where = "RUN"
        folder = latest_run
        status = _check_run_outputs(prefix, folder)
        print("Found in:", where, "->", folder)
        print(status)
        return

    print("Not found in MASTER, QC_TOP, or latest RUN folder.")


=== PPO ARTIFACT HEALTH CHECK (Updated) ===
Time: 2026-03-07 22:37:33
BASE: /content/drive/MyDrive/Results_May_2025
MASTER_DIR: /content/drive/MyDrive/Results_May_2025/ppo_models_master | exists: True
QC_DIR: /content/drive/MyDrive/Results_May_2025/ppo_models_QC_TOP | exists: True

[Latest Run]
Latest run folder: /content/drive/MyDrive/Results_May_2025/ppo_walkforward_results_20260307_223236
Latest run state: completed_or_partial_outputs_present
Latest summary file: /content/drive/MyDrive/Results_May_2025/ppo_walkforward_results_20260307_223236/summary_test_mode.csv
Latest manifest file: /content/drive/MyDrive/Results_May_2025/ppo_walkforward_results_20260307_223236/artifact_manifest.json
Manifest artifact prefixes: 4
Manifest sample prefixes: ['RUN_LEVEL', 'ppo_AAPL_window1', 'ppo_NVDA_window1', 'ppo_MSFT_window1']
Summary rows: 3
Tickers in summary: ['AAPL', 'MSFT', 'NVDA']

[Counts]
MASTER counts: {'zip': 3, 'pkl': 3, 'features': 3, 'prob_cfg': 3, 'info': 3, 'pred': 0, 'pred_compat

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
